# Thử chéo pair và cụm (Qwen3-4B + adapter fold 0)

GPU RTX Pro 6000, phiên tắt Internet. Chỉ mẫu pool; không có dữ liệu test.

In [ ]:
# ===== Cấu hình =====
CAU_HINH = {'so_toi_da': 600, 'seed': 43, 'gio_toi_da': 2.5}


In [ ]:
# Ghi mã vào working (phiên tắt Internet)
from pathlib import Path
Path('kaggle_train.py').write_text('"""Huấn luyện Qwen3-4B + LoRA trên Kaggle (RTX Pro 6000, phiên tắt Internet). Notebook nlp-group3 nhúng file này và gọi main(CAU_HINH).\n\nche_do="thu_b5" (B5): chạy so_buoc bước để kiểm môi trường: loss giảm, lưu/nạp lại checkpoint cho cùng loss, đo token/giây và bộ nhớ, sinh thử\nvài mẫu của fold kiểm bằng mô hình fine-tune và zero-shot. che_do="b6" (B6): dò loại dần trên 1 fold, xem chay_b6.\nche_do="b78" (B7, B8): cấu hình đã chốt chạy trên pair fold 0 và cụm fold 1, 2, chấm thêm LoRA attention, xem chay_b78.\nKHÓA TEST: chỉ đọc pool_*.jsonl.gz, dừng nếu thấy file test trong /kaggle/input.\nCác hàm thuần (xay_mau, chia_lo, lich_lr, chon_vong2, ...) không cần torch để test được trên máy (test_kaggle_train.py).\n"""\nimport gzip\nimport json\nimport math\nimport random\nimport subprocess\nimport sys\nimport time\nimport traceback\nfrom pathlib import Path\n\nVAO = Path("/kaggle/input")\nRA = Path("/kaggle/working")\nNHAC_EOS = "<|im_end|>"\n\n\n# ---------------------------------------------------------------- hàm thuần\ndef xay_mau(mau, tok, max_len):\n    """Mã hóa mỗi mẫu: prompt theo khuôn chat (tắt suy luận), đích kết thúc bằng <|im_end|>. Loss chỉ trên phần dịch (labels -100 cho prompt).\n    Trả (danh sách {id, ids, labels, fold5}, số mẫu vượt max_len bị bỏ)."""\n    ra, bo = [], 0\n    for m in mau:\n        p = tok.apply_chat_template(m["messages"][:1], tokenize=False, add_generation_prompt=True, enable_thinking=False)\n        a = m["messages"][1]["content"] + NHAC_EOS + "\\n"\n        ip, ia = tok(p, add_special_tokens=False)["input_ids"], tok(a, add_special_tokens=False)["input_ids"]\n        if len(ip) + len(ia) > max_len:\n            bo += 1\n            continue\n        ra.append({"id": m["id"], "ids": ip + ia, "labels": [-100] * len(ip) + ia, "fold5": m["fold5"],\n                   "huong": m.get("huong"), "loai": m.get("loai"), "category": m.get("category"), "product_id": m.get("product_id")})\n    return ra, bo\n\n\ndef cat_ngan_sach(k, micro_bs, ngan_sach):\n    """k đã xếp theo độ dài: cắt thành lô <= micro_bs mẫu và (số mẫu x mẫu dài nhất) <= ngan_sach token, tránh tràn GPU ở lô dài."""\n    lo, hien = [], []\n    for m in k:\n        if hien and (len(hien) >= micro_bs or (len(hien) + 1) * len(m["ids"]) > ngan_sach):\n            lo.append(hien)\n            hien = []\n        hien.append(m)\n    return lo + [hien] if hien else lo\n\n\ndef chia_lo(mau, micro_bs, seed, ngan_sach=8192, he_so_xo=50):\n    """Lô gần cùng độ dài để ít đệm: xáo, cắt khúc he_so_xo*micro_bs, xếp theo độ dài trong khúc, chia lô theo ngân sách, rồi xáo thứ tự lô."""\n    rng = random.Random(seed)\n    x = list(mau)\n    rng.shuffle(x)\n    lo = []\n    khuc = he_so_xo * micro_bs\n    for i in range(0, len(x), khuc):\n        lo += cat_ngan_sach(sorted(x[i:i + khuc], key=lambda m: len(m["ids"])), micro_bs, ngan_sach)\n    rng.shuffle(lo)\n    return lo\n\n\ndef gom_buoc(lo, co_batch):\n    """Gom micro-batch thành bước optimizer: mỗi bước gom tới khi đủ co_batch mẫu. Đếm trước để lịch lr có đúng tổng số bước (số mẫu mỗi bước có thể hơi vượt)."""\n    buoc, hien, dem = [], [], 0\n    for k in lo:\n        hien.append(k)\n        dem += len(k)\n        if dem >= co_batch:\n            buoc.append(hien)\n            hien, dem = [], 0\n    return buoc + [hien] if hien else buoc\n\n\ndef lich_lr(buoc, tong, ti_le_warmup=0.03):\n    """Hệ số lr: tăng tuyến tính trong warmup rồi giảm theo cosine về 0."""\n    w = max(1, int(tong * ti_le_warmup))\n    if buoc < w:\n        return (buoc + 1) / w\n    return 0.5 * (1 + math.cos(math.pi * (buoc - w) / max(1, tong - w)))\n\n\n# B6: 6 cấu hình của lưới + 1 cấu hình LoRA chỉ attention (cùng lr, rank với lr1e-4_r16, để biết có dùng được kiểu nhanh hơn không)\nCAU_HINH_B6 = [{"ten": f"lr{lr}_r{r}", "lr": float(lr), "rank": r} for r in (16, 64) for lr in ("5e-5", "1e-4", "2e-4")]\nCAU_HINH_B6.append({"ten": "lr1e-4_r16_attn", "lr": 1e-4, "rank": 16, "muc_tieu": ["q_proj", "k_proj", "v_proj", "o_proj"]})\nSO_EPOCH_B6 = 2       # tối đa 2 epoch (05/11/2026), các bước sau cũng dùng 2 epoch\n\n\nclass DungHetGio(Exception):\n    """Dừng có trật tự: đã lưu trạng thái, phiên sau chạy tiếp được."""\n\n\ndef diem_macro(loss_nhom):\n    """loss_nhom {\'huong|loai\': loss}. Mỗi hướng = trung bình đều title và mo_ta; điểm = trung bình đều các hướng (không theo token)."""\n    theo_huong = {}\n    for k, v in loss_nhom.items():\n        theo_huong.setdefault(k.split("|")[0], []).append(v)\n    return sum(sum(x) / len(x) for x in theo_huong.values()) / len(theo_huong)\n\n\ndef chon_vong2(diem, moi=0.01, toi_thieu=2, toi_da=3):\n    """diem {tên: loss, thấp là tốt}. Lấy toi_thieu cấu hình đầu, thêm cấu hình lệch không quá moi so với đầu bảng, trần toi_da.\n    Bỏ cấu hình loss không hữu hạn."""\n    x = sorted((v, k) for k, v in diem.items() if math.isfinite(v))\n    if not x:\n        raise RuntimeError("khong_con_cau_hinh_hop_le")\n    chon = [k for _, k in x[:toi_thieu]]\n    chon += [k for v, k in x[toi_thieu:toi_da] if v <= x[0][0] * (1 + moi)]\n    return chon\n\n\ndef lay_phan_tang(mau, moi_nhom, seed):\n    """Mẫu cố định theo (hướng, loại), dùng chung cho mọi cấu hình để so sánh theo cặp."""\n    rng = random.Random(seed)\n    nhom = {}\n    for m in sorted(mau, key=lambda m: m["id"]):\n        nhom.setdefault(f"{m[\'huong\']}|{m[\'loai\']}", []).append(m)\n    ra = []\n    for k in sorted(nhom):\n        ra += rng.sample(nhom[k], min(moi_nhom, len(nhom[k])))\n    return ra\n\n\ndef bootstrap_chenh(a, b, nhom, so_lan=2000, seed=42):\n    """a, b: điểm từng mục của hai cấu hình trên cùng mục; nhom: \'huong|loai\' từng mục. Khoảng 95% của chênh (a - b) theo điểm macro, lấy mẫu lại trong từng nhóm."""\n    rng = random.Random(seed)\n    theo = {}\n    for x, y, n in zip(a, b, nhom):\n        theo.setdefault(n, []).append(x - y)\n    kq = []\n    for _ in range(so_lan):\n        d = {n: sum(rng.choices(v, k=len(v))) / len(v) for n, v in theo.items()}\n        kq.append(diem_macro(d))\n    kq.sort()\n    return {"chenh": diem_macro({n: sum(v) / len(v) for n, v in theo.items()}), "thap": kq[int(0.025 * so_lan)], "cao": kq[int(0.975 * so_lan) - 1]}\n\n\n# B7, B8 (Nhật Anh duyệt 05/11/2026): cấu hình chốt từ B6; (tên việc, nhánh, fold kiểm). cum_f0 không huấn luyện, dùng lại adapter B6\nCAU_HINH_CHOT = {"ten": "lr2e-4_r64", "lr": 2e-4, "rank": 64}\nVIEC_B78 = [("cum_f0", "cum", 0), ("pair_f0", "pair", 0), ("cum_f1", "cum", 1), ("cum_f2", "cum", 2)]\nCAP_ATTN = ("lr1e-4_r16_attn", "lr1e-4_r16")\n\n\ndef chon_mau_sinh(mau_cum, mau_nhanh, fold, moi_nhom, seed, rut_gon=None):\n    """Mẫu sinh cố định của một fold: chọn trên nhánh cụm đúng như B6 (seed + 1), rồi lấy cùng id ở nhánh cần chấm, để hai nhánh chấm trên cùng sản phẩm.\n    rut_gon: giữ vài mẫu đầu mỗi nhóm khi chạy khói, vẫn là tập con của mẫu thật nên dùng lại được bản dịch đã có. Trả (mẫu, số id thiếu ở nhánh đó)."""\n    chon = lay_phan_tang([m for m in mau_cum if m["fold5"] == fold and m["phan"] == 0], moi_nhom, seed + 1)\n    if rut_gon:\n        dem, gon = {}, []\n        for m in chon:\n            k = f"{m[\'huong\']}|{m[\'loai\']}"\n            dem[k] = dem.get(k, 0) + 1\n            if dem[k] <= rut_gon:\n                gon.append(m)\n        chon = gon\n    theo = {m["id"]: m for m in mau_nhanh}\n    return [theo[m["id"]] for m in chon if m["id"] in theo], sum(m["id"] not in theo for m in chon)\n\n\ndef gop_loss(tung_cat):\n    """tung_cat {\'huong|loai|category\': (tổng loss, số token)}. Trả (loss {\'huong|loai\'} cộng theo token, điểm macro theo hướng)."""\n    gop = {}\n    for k, (t, n) in tung_cat.items():\n        a = gop.setdefault(k.rsplit("|", 1)[0], [0.0, 0])\n        a[0] += t\n        a[1] += n\n    nhom = {k: t / max(n, 1) for k, (t, n) in sorted(gop.items())}\n    return nhom, diem_macro(nhom)\n\n\n# Tỉ lệ title : mô tả (mục tiêu d, plan mục 0). Trọng số đặt theo mẫu: mô tả = 1, title = w. Một micro-batch: token nhân trọng số mẫu, chia tổng trọng số token.\n# Một bước: micro-batch nặng theo (tổng trọng số mẫu / số mẫu của bước) nhân hệ số chung he_chuan, KHÔNG chia cho tổng trọng số của bước: nhiều bước chỉ toàn title\n# hoặc toàn mô tả, chia theo bước sẽ triệt tiêu trọng số ở đó (đo thật: tỉ trọng title kẹt trong 14%–71%). w = 1 trùng đúng cách tính không trọng số.\ndef tom_tat_buoc(ds_buoc):\n    """Rút gọn để tính nhanh: mỗi bước là danh sách (số mẫu title, số mẫu mô tả, token đích title, token đích mô tả) từng micro-batch."""\n    ra = []\n    for buoc in ds_buoc:\n        hang = []\n        for k in buoc:\n            n_t = n_m = t_t = t_m = 0\n            for m in k:\n                so = sum(1 for x in m["labels"] if x != -100)\n                if m["loai"] == "title":\n                    n_t, t_t = n_t + 1, t_t + so\n                else:\n                    n_m, t_m = n_m + 1, t_m + so\n            hang.append((n_t, n_m, t_t, t_m))\n        ra.append(hang)\n    return ra\n\n\ndef ti_trong_title(tom_tat, w):\n    """Phần trọng số title trong loss, cộng trên mọi bước; là tỉ lệ danh nghĩa khi đặt w, chưa tính tác động của clip gradient."""\n    title = tong = 0.0\n    for hang in tom_tat:\n        so_mau = sum(a + b for a, b, _, _ in hang)\n        for a, b, c, d in hang:\n            nang = (w * a + b) / so_mau\n            mau = w * c + d\n            tong += nang\n            if mau:\n                title += nang * (w * c / mau)\n    return title / tong\n\n\ndef he_chuan(mau, w):\n    """Giữ độ lớn loss trung bình như khi không trọng số, để lr vẫn so được với mốc."""\n    return len(mau) / sum(trong_so_lo(mau, w))\n\n\ndef tim_w_title(tom_tat, p, so_lan=60):\n    """Tìm w để tỉ trọng title đạt p, chia đôi trên thang log vì ti_trong_title tăng theo w."""\n    if not 0 < p < 1:\n        raise ValueError(f"p_ngoai_khoang:{p}")\n    thap, cao = math.log(1e-4), math.log(1e4)\n    for _ in range(so_lan):\n        giua = (thap + cao) / 2\n        if ti_trong_title(tom_tat, math.exp(giua)) < p:\n            thap = giua\n        else:\n            cao = giua\n    return math.exp((thap + cao) / 2)\n\n\ndef trong_so_lo(k, w):\n    """Trọng số từng mẫu của micro-batch: title = w, mô tả = 1."""\n    return [w if m["loai"] == "title" else 1.0 for m in k]\n\n\ndef kiem_bat_thuong(cham, chrf_toi_thieu, lap_toi_da):\n    """Dấu hiệu lần chạy hỏng, để dừng sớm thay vì đốt tiếp quota khi không ai theo dõi. Trả lý do hoặc None. Ngưỡng là để bắt hỏng, không phải để chọn tỉ lệ."""\n    diem = cham["chrf"]["chrf_macro_huong"]\n    if diem is None or diem < chrf_toi_thieu:\n        return f"chrF {diem} duoi nguong {chrf_toi_thieu}"\n    ti_le = cham["sinh_lap"] / max(cham["so_mau"], 1)\n    if ti_le > lap_toi_da:\n        return f"sinh lap {ti_le:.3f} vuot nguong {lap_toi_da}"\n    return None\n\n\ndef chon_trang_thai(ds):\n    """ds: [(đường dẫn, trạng thái)]. Khi gắn cả bản lưu và đầu ra chính notebook, lấy bản tiến xa hơn (nhiều bước và nhiều phần chấm hơn), tránh nối nhầm bản cũ."""\n    def tien_do(x):\n        viec = x[1].get("viec", {}).values()\n        return (sum(1 for v in viec if v.get("cham")), sum(v.get("buoc", 0) for v in viec))\n    return max(ds, key=tien_do)\n\n\n# ---------------------------------------------------------------- môi trường\ndef tim_mot(mau_ten):\n    ds = sorted(VAO.rglob(mau_ten))\n    if not ds:\n        raise RuntimeError(f"khong_thay:{mau_ten}")\n    return ds[0]\n\n\ndef kiem_khoa_test():\n    la = [str(p) for p in VAO.rglob("*") if p.is_file() and ("test_" in p.name or p.name.startswith("do_khoa_") or p.name in {"split.parquet", "ds_test.txt"})\n          and "site-packages" not in str(p) and "arc-prize" not in str(p)]\n    if la:\n        raise RuntimeError(f"CO_FILE_TEST_TRONG_INPUT:{la[:5]}")\n\n\ndef kiem_khoa_test_do():\n    """Chế độ đo test (Nhật Anh duyệt 07/11/2026): được phép có file do_khoa_*.jsonl*; mọi file \'test_*\' khác, split.parquet, ds_test.txt vẫn là lỗi.\n    Phải có ít nhất một file do_khoa_*. Việc không rò sang pool kiểm riêng bằng kiem_ro_ri_test."""\n    la = [p for p in VAO.rglob("*") if p.is_file() and "site-packages" not in str(p) and "arc-prize" not in str(p)]\n    cam = [str(p) for p in la if p.name in {"split.parquet", "ds_test.txt"} or ("test_" in p.name and not p.name.startswith("do_khoa_"))]\n    if cam:\n        raise RuntimeError(f"CO_FILE_TEST_LA_TRONG_INPUT:{cam[:5]}")\n    if not any(p.name.startswith("do_khoa_") for p in la):\n        raise RuntimeError("THIEU_DO_KHOA")\n\n\ndef kiem_ro_ri_test(mau_test, mau_pool):\n    """Không sản phẩm hay id nào của test nằm trong pool (cùng nhánh). Trả (số sp test, số mẫu test)."""\n    sp_pool, id_pool = {m["product_id"] for m in mau_pool}, {m["id"] for m in mau_pool}\n    chung_sp = {m["product_id"] for m in mau_test} & sp_pool\n    chung_id = {m["id"] for m in mau_test} & id_pool\n    if chung_sp or chung_id:\n        raise RuntimeError(f"TEST_ROI_VAO_POOL:{len(chung_sp)}sp:{len(chung_id)}id")\n    return len({m["product_id"] for m in mau_test}), len(mau_test)\n\n\ndef cai_thu_vien():\n    wh = tim_mot("peft-*.whl").parent\n    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", str(wh), "peft", "sacrebleu"],\n                       capture_output=True, text=True)\n    print((r.stdout or "")[-1500:], (r.stderr or "")[-1500:], flush=True)\n    if r.returncode:\n        raise RuntimeError("cai_thu_vien_loi")\n\n\ndef doc_mau(nhanh):\n    """Kaggle có thể tự giải nén .gz khi nhận Dataset, nên đọc được cả hai dạng (nhận theo chữ ký file, không theo đuôi)."""\n    ds = [p for p in sorted(VAO.rglob(f"pool_{nhanh}.jsonl*")) if p.is_file()]\n    if not ds:\n        raise RuntimeError(f"khong_thay:pool_{nhanh}.jsonl")\n    with open(ds[0], "rb") as f:\n        nen = f.read(2) == b"\\x1f\\x8b"\n    print("doc mau:", ds[0], "nen" if nen else "khong nen", flush=True)\n    mo = gzip.open(ds[0], "rt", encoding="utf-8") if nen else open(ds[0], encoding="utf-8")\n    with mo as f:\n        return [json.loads(x) for x in f]\n\n\n# ---------------------------------------------------------------- phần cần torch\ndef lo_tensor(lo, tok, thiet_bi):\n    import torch\n    dai = max(len(m["ids"]) for m in lo)\n    pad = tok.pad_token_id\n    ids = torch.tensor([m["ids"] + [pad] * (dai - len(m["ids"])) for m in lo], device=thiet_bi)\n    lab = torch.tensor([m["labels"] + [-100] * (dai - len(m["labels"])) for m in lo], device=thiet_bi)\n    mask = torch.tensor([[1] * len(m["ids"]) + [0] * (dai - len(m["ids"])) for m in lo], device=thiet_bi)\n    return ids, lab, mask\n\n\ndef loss_kiem(model, tok, mau, micro_bs, thiet_bi, ngan_sach=8192):\n    """Loss trung bình theo token đích trên tập kiểm cố định."""\n    tong, dem = loss_tong(model, tok, mau, micro_bs, thiet_bi, ngan_sach)\n    return tong / max(dem, 1)\n\n\ndef loss_tong(model, tok, mau, micro_bs, thiet_bi, ngan_sach=8192):\n    """Tổng loss và số token đích, để cộng dồn đúng trọng số giữa các nhóm."""\n    import torch\n    import torch.nn.functional as F\n    model.eval()\n    tong, dem = 0.0, 0\n    with torch.no_grad():\n        for k in cat_ngan_sach(sorted(mau, key=lambda m: len(m["ids"])), micro_bs, ngan_sach):\n            ids, lab, mask = lo_tensor(k, tok, thiet_bi)\n            logit = model(input_ids=ids, attention_mask=mask).logits[:, :-1]\n            nhan = lab[:, 1:]\n            tong += F.cross_entropy(logit.reshape(-1, logit.size(-1)).float(), nhan.reshape(-1), ignore_index=-100, reduction="sum").item()\n            dem += (nhan != -100).sum().item()\n    model.train()\n    return tong, dem\n\n\ndef loss_co_trong_so(model, ids, lab, mask, ts, thiet_bi):\n    """Loss một micro-batch: token nhân trọng số mẫu, chia tổng trọng số token. ts toàn 1 thì bằng out.loss của mô hình (có phép thử trước khi chạy)."""\n    import torch\n    import torch.nn.functional as F\n    logit = model(input_ids=ids, attention_mask=mask).logits[:, :-1].float()\n    nhan = lab[:, 1:]\n    ce = F.cross_entropy(logit.reshape(-1, logit.size(-1)), nhan.reshape(-1), ignore_index=-100, reduction="none").view_as(nhan)\n    tw = torch.tensor(ts, dtype=torch.float32, device=thiet_bi)[:, None] * (nhan != -100)\n    return (ce * tw).sum() / tw.sum()\n\n\ndef nap_goc(duong_dan, attn="sdpa"):\n    """attn mặc định sdpa như mọi lần chạy cũ (Qwen3); mô hình BLOOM (EcomGPT) không dùng sdpa nên truyền attn="eager"."""\n    import torch\n    from transformers import AutoModelForCausalLM\n    return AutoModelForCausalLM.from_pretrained(str(duong_dan), dtype=torch.bfloat16, device_map={"": 0}, attn_implementation=attn)\n\n\ndef chon_tran_sinh(n_tok_nguon):\n    """Trần max_new_tokens theo độ dài NGUỒN (đo trên pool 07/11/2026: token đích / token nguồn lớn nhất 2,94 ở title zh->vi, 1,59 ở mô tả), nên 3,5 lần cộng 64.\n    Dùng khi đo test, vì lúc dịch thật không có tham chiếu để lấy độ dài."""\n    return min(2048, math.ceil(3.5 * n_tok_nguon) + 64)\n\n\ndef sinh(model, tok, mau, thiet_bi, bs=32, ngan_sach=32768, tran_theo="dich", lam_prompt=None):\n    """Giải mã greedy cố định; trần max_new_tokens theo độ dài đích tham chiếu (tran_theo="dich", mặc định, giữ nguyên cho mọi lần chạy đã làm)\n    hoặc theo độ dài nguồn (tran_theo="nguon", dùng khi đo test). Xếp theo độ dài và cắt lô theo ngân sách cho nhanh; trả đúng thứ tự mau."""\n    import torch\n    tok.padding_side = "left"\n    model.eval()\n    mo = []\n    for m in mau:\n        # lam_prompt=None: khuôn chat Qwen3 như mọi lần chạy cũ; mô hình khác truyền hàm dựng lời dẫn riêng (xem kaggle_quota.tao_khuon)\n        p = lam_prompt(m) if lam_prompt else tok.apply_chat_template(m["messages"][:1], tokenize=False, add_generation_prompt=True, enable_thinking=False)\n        if tran_theo == "nguon":\n            tran = chon_tran_sinh(len(tok(m["nguon"], add_special_tokens=False)["input_ids"]))\n        else:\n            tran = min(1500, len(tok(m["dich"], add_special_tokens=False)["input_ids"]) * 2 + 64)\n        mo.append({"m": m, "p": p, "tran": tran, "ids": [0] * (len(tok(p, add_special_tokens=False)["input_ids"]) + tran)})\n    kq = {}\n    for k in cat_ngan_sach(sorted(mo, key=lambda x: len(x["ids"])), bs, ngan_sach):\n        enc = tok([x["p"] for x in k], return_tensors="pt", padding=True, add_special_tokens=False).to(thiet_bi)\n        with torch.no_grad():\n            out = model.generate(**enc, max_new_tokens=max(x["tran"] for x in k), do_sample=False, pad_token_id=tok.pad_token_id)\n        for x, o in zip(k, out):\n            kq[x["m"]["id"]] = tok.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True)\n    tok.padding_side = "right"\n    return [{"id": m["id"], "hyp_tho": kq[m["id"]], "hyp": kq[m["id"]]} for m in mau]\n\n\ndef loss_nhom(model, tok, mau, thiet_bi, ngan_sach=8192):\n    """Loss theo token đích của từng nhóm \'huong|loai\' trên mẫu kiểm cố định; trả dict và điểm macro."""\n    theo = {}\n    for m in mau:\n        theo.setdefault(f"{m[\'huong\']}|{m[\'loai\']}", []).append(m)\n    ra = {k: loss_kiem(model, tok, v, 32, thiet_bi, ngan_sach) for k, v in sorted(theo.items())}\n    return ra, diem_macro(ra)\n\n\ndef chay_b6(cfg, bc):\n    """B6 dò loại dần trên 1 fold (Nhật Anh chốt 05/11/2026). Vòng 1: mọi cấu hình 1 epoch, so loss. Vòng 2: cấu hình được chọn chạy nốt tới epoch cuối (SO_EPOCH_B6).\n    Vòng 3: cấu hình được chọn, ở epoch loss tốt nhất, sinh và chấm chrF trên mẫu cố định. Lịch lr cosine tính cho toàn bộ SO_EPOCH_B6 epoch nên vòng 2 nối tiếp đúng lịch.\n    Trạng thái ghi vào b6/b6_trang_thai.json sau mỗi mốc; hết giờ thì lưu rồi dừng có trật tự, phiên sau chạy tiếp."""\n    import shutil\n    import torch\n    import transformers\n    import peft\n    from peft import LoraConfig, PeftModel, get_peft_model\n    from transformers import AutoTokenizer\n\n    thiet_bi = "cuda:0"\n    t_bat_dau = cfg["_t_bat_dau"]\n    het_giay = cfg.get("gio_toi_da", 8.5) * 3600\n    thu_ra = RA / "b6"\n    tep = thu_ra / "b6_trang_thai.json"\n    if not tep.exists():                               # phiên sau: gắn trạng thái phiên trước làm input\n        cu = sorted(VAO.rglob("b6_trang_thai.json"))\n        if cu:\n            shutil.copytree(cu[0].parent, thu_ra)\n            print("chay tiep tu:", cu[0], flush=True)\n    thu_ra.mkdir(exist_ok=True)\n    st = json.loads(tep.read_text(encoding="utf-8")) if tep.exists() else {"cau_hinh": {}, "chon": None, "cham": {}}\n\n    def ghi():\n        tam = tep.with_suffix(".tmp")\n        tam.write_text(json.dumps(st, ensure_ascii=False, indent=1, default=str), encoding="utf-8")\n        tam.replace(tep)\n        bc["b6"] = st\n\n    bc["moi_truong"] = {"torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__, "gpu": torch.cuda.get_device_name(0)}\n    thu_mh = tim_mot("REVISION.txt").parent\n    bc["mo_hinh"] = thu_mh.joinpath("REVISION.txt").read_text().split()\n    tok = AutoTokenizer.from_pretrained(str(thu_mh))\n    if tok.pad_token_id is None:\n        tok.pad_token = tok.eos_token\n    tok.padding_side = "right"\n    goc_mau = doc_mau(cfg["nhanh"])\n    ma, bo = xay_mau(goc_mau, tok, cfg["max_len"])\n    fk = cfg["fold_kiem"]\n    train = [m for m in ma if m["fold5"] != fk]\n    kiem = lay_phan_tang([m for m in ma if m["fold5"] == fk], cfg["so_kiem_moi_nhom"], cfg["seed"])\n    if cfg.get("so_train"):\n        train = random.Random(cfg["seed"]).sample(train, cfg["so_train"])\n    if {m["id"] for m in train} & {m["id"] for m in kiem}:\n        raise RuntimeError("giao_train_kiem")\n    st.update(so_train=len(train), so_kiem=len(kiem), so_mau_bo_max_len=bo, so_token_train=sum(len(m["ids"]) for m in train), fold_kiem=fk)\n    co_batch = cfg["micro_bs"] * cfg["tich_luy"]\n    # Chống nối nhầm trạng thái: phiên sau phải cùng dữ liệu, cùng cách chia lô, cùng số epoch với phiên đã tạo trạng thái\n    van_tay = {"so_train": len(train), "so_kiem": len(kiem), "so_token_train": st["so_token_train"], "fold_kiem": fk, "seed": cfg["seed"], "so_epoch": SO_EPOCH_B6,\n               "micro_bs": cfg["micro_bs"], "ngan_sach": cfg["ngan_sach_token"], "nhanh": cfg["nhanh"]}\n    if st.get("van_tay") and st["van_tay"] != van_tay:\n        raise RuntimeError(f"TRANG_THAI_KHONG_KHOP: cu {st[\'van_tay\']} moi {van_tay}")\n    st["van_tay"] = van_tay\n    # Bước của từng epoch được dựng trước, cùng seed cho mọi cấu hình; lịch lr dùng đúng tổng số bước của mọi epoch\n    buoc_theo_epoch = [gom_buoc(chia_lo(train, cfg["micro_bs"], cfg["seed"] + e, cfg["ngan_sach_token"]), co_batch) for e in range(SO_EPOCH_B6)]\n    tong_buoc = sum(len(x) for x in buoc_theo_epoch)\n    st["tong_buoc"] = tong_buoc\n    ds_cau_hinh = {c["ten"]: c for c in CAU_HINH_B6 if not cfg.get("chi_cau_hinh") or c["ten"] in cfg["chi_cau_hinh"]}\n    for ten in ds_cau_hinh:\n        st["cau_hinh"].setdefault(ten, {"epoch_xong": 0, "buoc": 0, "vi_tri_lo": 0, "loss": {}, "diem": {}, "token_moi_giay": {}})\n    ghi()\n    goc = nap_goc(thu_mh)\n    goc.config.use_cache = False\n    dem_dung = {"con": cfg.get("dung_sau_buoc")}       # chỉ dùng khi chạy khói, để thử chạy tiếp\n\n    def het_gio():\n        return time.time() - t_bat_dau > het_giay\n\n    def huan_luyen_toi(ten, den_epoch):\n        nonlocal goc\n        c, s = ds_cau_hinh[ten], st["cau_hinh"][ten]\n        if s["epoch_xong"] >= den_epoch or s.get("hong"):\n            return\n        d, moi_nhat = thu_ra / ten, thu_ra / ten / "moi_nhat"\n        d.mkdir(exist_ok=True)\n        torch.manual_seed(cfg["seed"])\n        if (moi_nhat / "opt.pt").exists():\n            m = PeftModel.from_pretrained(goc, str(moi_nhat), is_trainable=True)\n        else:\n            m = get_peft_model(goc, LoraConfig(r=c["rank"], lora_alpha=2 * c["rank"], lora_dropout=0.05, target_modules=c.get("muc_tieu", "all-linear"), task_type="CAUSAL_LM"))\n        tham_so = [p for p in m.parameters() if p.requires_grad]\n        opt = torch.optim.AdamW(tham_so, lr=c["lr"], weight_decay=0.0)\n        if (moi_nhat / "opt.pt").exists():\n            opt.load_state_dict(torch.load(moi_nhat / "opt.pt", map_location=thiet_bi))\n        s["so_tham_so"] = sum(p.numel() for p in tham_so)\n\n        def luu():\n            if moi_nhat.exists():\n                shutil.rmtree(moi_nhat)\n            m.save_pretrained(str(moi_nhat))\n            torch.save(opt.state_dict(), moi_nhat / "opt.pt")\n            ghi()\n\n        m.train()\n        hong = False\n        for e in range(s["epoch_xong"], den_epoch):\n            ds_buoc = buoc_theo_epoch[e]\n            i, tok_do, t0 = s["vi_tri_lo"], 0, time.time()\n            while i < len(ds_buoc) and not hong:\n                cac_lo = ds_buoc[i]\n                i += 1\n                so_mau = sum(len(k) for k in cac_lo)\n                tong = 0.0\n                for k in cac_lo:\n                    ids, lab, mask = lo_tensor(k, tok, thiet_bi)\n                    out = m(input_ids=ids, attention_mask=mask, labels=lab)\n                    (out.loss * len(k) / so_mau).backward()         # trọng số theo số mẫu thật của bước\n                    tong += out.loss.item() * len(k) / so_mau\n                    tok_do += int(mask.sum().item())\n                for g in opt.param_groups:\n                    g["lr"] = c["lr"] * lich_lr(s["buoc"], tong_buoc)\n                torch.nn.utils.clip_grad_norm_(tham_so, 1.0)\n                opt.step()\n                opt.zero_grad(set_to_none=True)\n                s["buoc"] += 1\n                s["vi_tri_lo"] = i\n                if not math.isfinite(tong):\n                    hong = True\n                if s["buoc"] % 100 == 0:\n                    print(ten, "epoch", e + 1, "buoc", s["buoc"], "/", tong_buoc, "loss", round(tong, 4), "lr", opt.param_groups[0]["lr"], flush=True)\n                if dem_dung["con"] is not None:\n                    dem_dung["con"] -= 1\n                dung = dem_dung["con"] == 0\n                if not hong and (het_gio() or dung):\n                    if dung:\n                        dem_dung["con"] = None\n                    luu()\n                    print("DUNG, da luu", ten, "epoch", e + 1, "vi tri lo", i, flush=True)\n                    goc = m.unload()           # gỡ LoRA khỏi mô hình gốc, tránh chồng hai lớp khi chạy tiếp trong cùng phiên\n                    del m, opt\n                    torch.cuda.empty_cache()\n                    raise DungHetGio(ten)\n            if hong:\n                s.update(hong=True)\n                s["diem"][str(e + 1)] = float("inf")\n                print("LOSS KHONG HUU HAN, bo cau hinh", ten, flush=True)\n                break\n            s["token_moi_giay"][str(e + 1)] = round(tok_do / max(time.time() - t0, 1e-6), 1)\n            s["epoch_xong"], s["vi_tri_lo"] = e + 1, 0\n            ln, dm = loss_nhom(m, tok, kiem, thiet_bi)\n            m.train()\n            s["loss"][str(e + 1)], s["diem"][str(e + 1)] = ln, dm\n            m.save_pretrained(str(d / f"ep{e + 1}"))\n            luu()\n            print(ten, "XONG epoch", e + 1, "diem loss", round(dm, 5), ln, "token/giay", s["token_moi_giay"][str(e + 1)], "gio da chay", round((time.time() - t_bat_dau) / 3600, 2), flush=True)\n        goc = m.unload()\n        del m, opt\n        torch.cuda.empty_cache()\n        ghi()\n\n    # Vòng 1\n    for ten in ds_cau_hinh:\n        huan_luyen_toi(ten, 1)\n    if st["chon"] is None:\n        st["chon"] = chon_vong2({t: s["diem"].get("1", float("inf")) for t, s in st["cau_hinh"].items()})\n        ghi()\n    print("CHON VONG 2:", st["chon"], flush=True)\n    # Vòng 2\n    for ten in st["chon"]:\n        huan_luyen_toi(ten, SO_EPOCH_B6)\n    # Vòng 3: sinh và chấm trên mẫu cố định, ở epoch loss tốt nhất của từng cấu hình\n    import pandas as pd\n    import cham_diem as D\n    mau_sinh = lay_phan_tang([m for m in goc_mau if m["fold5"] == fk and m["phan"] == 0], cfg["so_sinh_moi_nhom"], cfg["seed"] + 1)\n    tc = pd.DataFrame(mau_sinh)\n    diem_muc = {}\n    for ten in st["chon"]:\n        s = st["cau_hinh"][ten]\n        if s.get("hong"):\n            continue\n        ep = min(s["diem"], key=lambda e: s["diem"][e])\n        tep_sinh = thu_ra / ten / f"sinh_ep{ep}.jsonl"\n        if ten not in st["cham"]:\n            if het_gio():\n                raise DungHetGio("sinh")\n            m = PeftModel.from_pretrained(goc, str(thu_ra / ten / f"ep{ep}"))\n            t = time.time()\n            du = sinh(m, tok, mau_sinh, thiet_bi)\n            giay = time.time() - t\n            tep_sinh.write_text("\\n".join(json.dumps(r, ensure_ascii=False) for r in du) + "\\n", encoding="utf-8")\n            goc = m.unload()\n            del m\n            torch.cuda.empty_cache()\n            tm, th = D.cham(tc, pd.DataFrame(du), nhanh=cfg["nhanh"])\n            st["cham"][ten] = {"epoch": ep, "giay_moi_mau": round(giay / len(du), 3), "chrf": th["diem_chinh"], "theo_nhom": th.get("theo_nhom"),\n                               "ti_le_cat_loi_dan": float(tm.da_cat_loi_dan.mean()), "sinh_lap": int(tm.sinh_lap.sum()) if "sinh_lap" in tm else None}\n            ghi()\n            print(ten, "epoch", ep, "chrF", th["diem_chinh"].get("chrf_macro_huong"), "giay/mau", st["cham"][ten]["giay_moi_mau"], flush=True)\n        du = [json.loads(x) for x in tep_sinh.read_text(encoding="utf-8").splitlines()]\n        tm, _ = D.cham(tc, pd.DataFrame(du), nhanh=cfg["nhanh"])\n        diem_muc[ten] = tm.set_index("id").chrf\n    # So theo cặp: cấu hình đầu bảng theo chrF với từng cấu hình còn lại, khoảng tin cậy bootstrap (chứa 0 nghĩa là hòa)\n    xep = sorted(st["cham"], key=lambda t: -st["cham"][t]["chrf"]["chrf_macro_huong"])\n    nhom = [f"{m[\'huong\']}|{m[\'loai\']}" for m in mau_sinh]\n    ids = [m["id"] for m in mau_sinh]\n    st["so_cap_chrf"] = {f"{xep[0]} - {t}": bootstrap_chenh(diem_muc[xep[0]][ids].tolist(), diem_muc[t][ids].tolist(), nhom) for t in xep[1:]}\n    st["xep_hang_chrf"] = xep\n    st["xong"] = True\n    ghi()\n    bc["dat"] = True\n    print("XEP HANG CHRF:", xep, st["so_cap_chrf"], flush=True)\n\n\ndef chay_b78(cfg, bc):\n    """B7, B8 (D18): cấu hình đã chốt chạy trên pair fold 0 và cụm fold 1, 2; cụm fold 0 dùng lại adapter và bản dịch B6. Mỗi việc: huấn luyện, loss cả fold theo\n    hướng, loại, category, rồi sinh và chấm chrF trên 1.000 mẫu cố định. Trước đó chấm chrF LoRA attention so với LoRA mọi lớp cùng lr, rank ở epoch 1.\n    Không tự chạy thêm khi chốt chặn bật: tong_hop_b78.py tính chốt chặn trên máy, người quyết. Không dùng chung vòng huấn luyện với chay_b6 để giữ nguyên mã đã tạo ra kết quả B6.\n    Mỗi pha tự bỏ qua nếu đã có trong trạng thái; hết giờ thì lưu rồi dừng có trật tự, phiên sau chạy tiếp."""\n    import shutil\n    import pandas as pd\n    import torch\n    import transformers\n    import peft\n    from peft import LoraConfig, PeftModel, get_peft_model\n    from transformers import AutoTokenizer\n    import cham_diem as D\n\n    thiet_bi = "cuda:0"\n    t_bat_dau = cfg["_t_bat_dau"]\n    het_giay = cfg["gio_toi_da"] * 3600\n    ten_tt = cfg.get("ten_trang_thai", "b78")      # khảo sát tỉ lệ dùng trạng thái riêng "tl", không lẫn với B7, B8\n    thu_ra = RA / ten_tt\n    tep = thu_ra / f"{ten_tt}_trang_thai.json"\n    if not tep.exists():\n        cu = sorted(VAO.rglob(tep.name))\n        if cu:\n            duong_cu, _ = chon_trang_thai([(p, json.loads(p.read_text(encoding="utf-8"))) for p in cu])\n            shutil.copytree(duong_cu.parent, thu_ra)\n            print("chay tiep tu:", duong_cu, flush=True)\n    thu_ra.mkdir(exist_ok=True)\n    st = json.loads(tep.read_text(encoding="utf-8")) if tep.exists() else {"viec": {}, "attn": {}, "xong": None}\n    ds_viec = cfg.get("viec") or [{"ten": t, "nhanh": n, "fold": f, "ca_fold": True} for t, n, f in VIEC_B78]\n\n    def ghi():\n        tam = tep.with_suffix(".tmp")\n        tam.write_text(json.dumps(st, ensure_ascii=False, indent=1, default=str), encoding="utf-8")\n        tam.replace(tep)\n        bc["b78"] = st\n\n    def kiem_gio(o_dau):\n        if time.time() - t_bat_dau > het_giay:\n            raise DungHetGio(o_dau)\n\n    # Chống nối nhầm trạng thái: phiên sau phải cùng cách chạy với phiên đã tạo trạng thái\n    van_tay = {k: cfg.get(k) for k in ("seed", "micro_bs", "ngan_sach_token", "max_len", "so_kiem_moi_nhom", "so_sinh_moi_nhom", "rut_gon_sinh", "so_train", "so_mau_ca_fold")}\n    van_tay.update(so_epoch=SO_EPOCH_B6, cau_hinh=CAU_HINH_CHOT["ten"])\n    if st.get("van_tay") and st["van_tay"] != van_tay:\n        raise RuntimeError(f"TRANG_THAI_KHONG_KHOP: cu {st[\'van_tay\']} moi {van_tay}")\n    st["van_tay"] = van_tay\n\n    kho, st_b6 = None, None\n    if not cfg.get("viec"):\n        ds_b6 = sorted(VAO.rglob("b6_trang_thai.json"))     # kho lưu B6, notebook riêng nên không bị thay khi đẩy lại nlp-group3\n        if len(ds_b6) != 1:\n            raise RuntimeError(f"can_dung_mot_kho_luu_b6:{ds_b6}")\n        kho = ds_b6[0].parent\n        st_b6 = json.loads(ds_b6[0].read_text(encoding="utf-8"))\n        print("kho luu B6:", kho, flush=True)\n    bc["moi_truong"] = {"torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__, "gpu": torch.cuda.get_device_name(0)}\n    thu_mh = tim_mot("REVISION.txt").parent\n    bc["mo_hinh"] = thu_mh.joinpath("REVISION.txt").read_text().split()\n    tok = AutoTokenizer.from_pretrained(str(thu_mh))\n    if tok.pad_token_id is None:\n        tok.pad_token = tok.eos_token\n    tok.padding_side = "right"\n    goc_mau = {n: doc_mau(n) for n in ("cum", "pair")}\n    ma = {}\n    for n, g in goc_mau.items():\n        ma[n], bo = xay_mau(g, tok, cfg["max_len"])\n        st[f"so_mau_bo_max_len_{n}"] = bo\n    ghi()\n    goc = nap_goc(thu_mh)\n    goc.config.use_cache = False\n    dem_dung = {"con": cfg.get("dung_sau_buoc")}       # chỉ dùng khi chạy khói, để thử chạy tiếp\n\n    if any(j.get("p_title") for j in ds_viec):         # phép thử trước khi tốn giờ: trọng số 1:1 phải đúng bằng loss của mô hình, trên lô trộn title và mô tả\n        n_thu = ds_viec[0]["nhanh"]\n        tit = min((m for m in ma[n_thu] if m["loai"] == "title"), key=lambda m: len(m["ids"]))\n        mt = min((m for m in ma[n_thu] if m["loai"] == "mo_ta"), key=lambda m: len(m["ids"]))\n        ids, lab, mask = lo_tensor([tit, mt], tok, thiet_bi)\n        goc.eval()\n        with torch.no_grad():\n            a = goc(input_ids=ids, attention_mask=mask, labels=lab).loss.item()\n            b = loss_co_trong_so(goc, ids, lab, mask, [1.0, 1.0], thiet_bi).item()\n        goc.train()\n        st["thu_loss_1_1"] = {"out_loss": a, "co_trong_so": b}\n        print("thu loss 1:1", a, b, flush=True)\n        if not abs(a - b) <= 1e-3:         # viết thế này để NaN cũng bị bắt\n            raise RuntimeError(f"LOSS_TRONG_SO_KHONG_KHOP:{a}:{b}")\n\n    def dung_bat_thuong(ten, ly_do):\n        st["bat_thuong"] = f"{ten}: {ly_do}"\n        ghi()\n        print("BAT THUONG, dung phien de khong dot quota:", st["bat_thuong"], flush=True)\n        raise DungHetGio("bat_thuong")\n\n    def lay_mau_sinh(nhanh, fold):\n        mau, thieu = chon_mau_sinh(goc_mau["cum"], goc_mau[nhanh], fold, cfg["so_sinh_moi_nhom"], cfg["seed"], cfg.get("rut_gon_sinh"))\n        return mau, thieu\n\n    def tap_da_thay(nhanh, fold):\n        return D.xay_tap_da_thay(pd.DataFrame(goc_mau[nhanh])[["loai", "category", "nguon", "nhanh", "fold5"]], fold, nhanh)\n\n    def doc_sinh(duong_dan, mau):\n        ids = {m["id"] for m in mau}\n        du = [r for r in map(json.loads, duong_dan.read_text(encoding="utf-8").splitlines()) if r["id"] in ids]\n        if len(du) != len(mau):\n            raise RuntimeError(f"ban_dich_thieu_muc:{duong_dan}:{len(du)}/{len(mau)}")\n        return du\n\n    def ghi_sinh(duong_dan, du):\n        tam = duong_dan.with_suffix(".tmp")\n        tam.write_text("\\n".join(json.dumps(r, ensure_ascii=False) for r in du) + "\\n", encoding="utf-8")\n        tam.replace(duong_dan)\n\n    def cham_mot(nhanh, mau, du, tap, epoch, giay_moi_mau):\n        tm, th = D.cham(pd.DataFrame(mau), pd.DataFrame(du), nhanh=nhanh, tap_da_thay=tap)\n        return tm, D._sach({"epoch": str(epoch), "giay_moi_mau": giay_moi_mau, "so_mau": len(mau), "sinh_lap": int(tm.sinh_lap.sum()), "ti_le_cat_loi_dan": float(tm.da_cat_loi_dan.mean()),\n                            "chrf": th["diem_chinh"], "theo_huong": th["theo_huong"], "theo_nhom": th["theo_nhom"], "theo_category": th["theo_category"], "dong_da_thay_moi": th.get("dong_da_thay_moi")})\n\n    def huan_luyen(ten, v, nhanh, fold, p_title=None, lr=None, loai_sp=None):\n        """fold là số: huấn luyện trên các fold khác và kiểm trên fold đó (như mọi lần chạy trước). fold là None: BẢN CUỐI trên TOÀN pool (Nhật Anh duyệt 07/11/2026, Prompt 808);\n        loai_sp (tập product_id) = bỏ mẫu title của các sản phẩm đó khỏi tập huấn luyện, giữ nguyên mẫu mô tả; mẫu \'kiểm\' lấy từ chính tập huấn luyện nên CHỈ để bắt hỏng, không phải đo chất lượng."""\n        nonlocal goc\n        if v["epoch_xong"] >= SO_EPOCH_B6 or v.get("hong"):\n            return\n        if fold is None:\n            tat_ca = [m for m in ma[nhanh] if not (loai_sp and m["loai"] == "title" and m["product_id"] in loai_sp)]\n            v.update(toan_pool=True, so_mau_pool=len(ma[nhanh]), so_title_bi_loai=len(ma[nhanh]) - len(tat_ca), kiem_la_train=True)\n            train = tat_ca\n        else:\n            train = [m for m in ma[nhanh] if m["fold5"] != fold]\n        if cfg.get("so_train"):\n            train = random.Random(cfg["seed"]).sample(train, cfg["so_train"])\n        if fold is None:\n            kiem = lay_phan_tang(train, cfg["so_kiem_moi_nhom"], cfg["seed"])\n        else:\n            kiem = lay_phan_tang([m for m in ma[nhanh] if m["fold5"] == fold], cfg["so_kiem_moi_nhom"], cfg["seed"])\n            if {m["id"] for m in train} & {m["id"] for m in kiem}:\n                raise RuntimeError("giao_train_kiem")\n        co_batch = cfg["micro_bs"] * cfg["tich_luy"]\n        buoc_theo_epoch = [gom_buoc(chia_lo(train, cfg["micro_bs"], cfg["seed"] + e, cfg["ngan_sach_token"]), co_batch) for e in range(SO_EPOCH_B6)]\n        tong_buoc = sum(len(x) for x in buoc_theo_epoch)\n        if v.get("tong_buoc") not in (None, tong_buoc):\n            raise RuntimeError(f"TONG_BUOC_KHONG_KHOP:{ten}:{v[\'tong_buoc\']}:{tong_buoc}")\n        v.update(so_train=len(train), so_kiem=len(kiem), so_token_train=sum(len(m["ids"]) for m in train), tong_buoc=tong_buoc)\n        w = None\n        if p_title:\n            tat = tom_tat_buoc([b for e in buoc_theo_epoch for b in e])\n            w = tim_w_title(tat, p_title)\n            if v.get("w_title") is not None and abs(v["w_title"] - w) > 1e-6:\n                raise RuntimeError(f"W_TITLE_KHONG_KHOP:{ten}:{v[\'w_title\']}:{w}")\n            chuan = he_chuan(train, w)\n            v.update(p_muc_tieu=p_title, w_title=w, he_chuan=chuan, p_thuc=ti_trong_title(tat, w), p_khi_w_1=ti_trong_title(tat, 1.0))\n            print(ten, "p muc tieu", p_title, "p thuc", round(v["p_thuc"], 5), "w_title", round(w, 5), "he_chuan", round(chuan, 4), "p khi w=1", round(v["p_khi_w_1"], 5), flush=True)\n            if abs(v["p_thuc"] - p_title) > 1e-3:\n                raise RuntimeError(f"KHONG_DAT_TI_LE:{ten}:{v[\'p_thuc\']}:{p_title}")\n        ghi()\n        c = {**CAU_HINH_CHOT, "lr": lr or CAU_HINH_CHOT["lr"]}\n        v["lr"] = c["lr"]\n        d, moi_nhat = thu_ra / ten, thu_ra / ten / "moi_nhat"\n        d.mkdir(exist_ok=True)\n        torch.manual_seed(cfg["seed"])\n        if v["buoc"] > 0 and not (moi_nhat / "opt.pt").exists():     # đã chạy dở mà mất checkpoint: dừng, không âm thầm tạo LoRA mới rồi chạy tiếp từ giữa\n            raise RuntimeError(f"MAT_CHECKPOINT:{ten}:buoc={v[\'buoc\']}")\n        if (moi_nhat / "opt.pt").exists():\n            m = PeftModel.from_pretrained(goc, str(moi_nhat), is_trainable=True)\n        else:\n            m = get_peft_model(goc, LoraConfig(r=c["rank"], lora_alpha=2 * c["rank"], lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM"))\n        tham_so = [p for p in m.parameters() if p.requires_grad]\n        opt = torch.optim.AdamW(tham_so, lr=c["lr"], weight_decay=0.0)\n        if (moi_nhat / "opt.pt").exists():\n            opt.load_state_dict(torch.load(moi_nhat / "opt.pt", map_location=thiet_bi))\n        v["so_tham_so"] = sum(p.numel() for p in tham_so)\n\n        def luu():\n            if moi_nhat.exists():\n                shutil.rmtree(moi_nhat)\n            m.save_pretrained(str(moi_nhat))\n            torch.save(opt.state_dict(), moi_nhat / "opt.pt")\n            ghi()\n\n        m.train()\n        for e in range(v["epoch_xong"], SO_EPOCH_B6):\n            ds_buoc = buoc_theo_epoch[e]\n            i, tok_do, t0 = v["vi_tri_lo"], 0, time.time()\n            while i < len(ds_buoc):\n                cac_lo = ds_buoc[i]\n                i += 1\n                so_mau = sum(len(k) for k in cac_lo)\n                tong = 0.0\n                for k in cac_lo:\n                    ids, lab, mask = lo_tensor(k, tok, thiet_bi)\n                    if w:\n                        ts = trong_so_lo(k, w)\n                        loss_k, he = loss_co_trong_so(m, ids, lab, mask, ts, thiet_bi), chuan * sum(ts) / so_mau\n                    else:\n                        loss_k, he = m(input_ids=ids, attention_mask=mask, labels=lab).loss, len(k) / so_mau\n                    (loss_k * he).backward()\n                    tong += loss_k.item() * he\n                    tok_do += int(mask.sum().item())\n                for g in opt.param_groups:\n                    g["lr"] = c["lr"] * lich_lr(v["buoc"], tong_buoc)\n                chuan_grad = float(torch.nn.utils.clip_grad_norm_(tham_so, 1.0))\n                # Clip có thể làm giảm tác dụng của trọng số: ghi tỉ lệ bước bị cắt theo loại để đánh giá, tách bước toàn title, toàn mô tả, trộn\n                nhom_buoc = {x["loai"] for k in cac_lo for x in k}\n                loai_buoc = "tron" if len(nhom_buoc) > 1 else next(iter(nhom_buoc))\n                tk = v.setdefault("thong_ke_grad", {}).setdefault(loai_buoc, {"so_buoc": 0, "tong_chuan": 0.0, "so_bi_cat": 0})\n                tk["so_buoc"] += 1\n                tk["tong_chuan"] += chuan_grad if math.isfinite(chuan_grad) else 0.0\n                tk["so_bi_cat"] += chuan_grad > 1.0\n                opt.step()\n                opt.zero_grad(set_to_none=True)\n                v["buoc"] += 1\n                v["vi_tri_lo"] = i\n                if not math.isfinite(tong):\n                    v["hong"] = True\n                    print("LOSS KHONG HUU HAN, bo viec", ten, "buoc", v["buoc"], flush=True)\n                    goc = m.unload()\n                    del m, opt\n                    torch.cuda.empty_cache()\n                    ghi()\n                    return\n                if v["buoc"] % 100 == 0:\n                    print(ten, "epoch", e + 1, "buoc", v["buoc"], "/", tong_buoc, "loss", round(tong, 4), "lr", opt.param_groups[0]["lr"],\n                          "grad", {a: (x["so_buoc"], round(x["tong_chuan"] / x["so_buoc"], 3), x["so_bi_cat"]) for a, x in v["thong_ke_grad"].items()}, flush=True)\n                if dem_dung["con"] is not None:\n                    dem_dung["con"] -= 1\n                dung = dem_dung["con"] == 0\n                if dung or time.time() - t_bat_dau > het_giay:\n                    if dung:\n                        dem_dung["con"] = None\n                    luu()\n                    print("DUNG, da luu", ten, "epoch", e + 1, "vi tri lo", i, flush=True)\n                    raise DungHetGio(ten)\n            v["token_moi_giay"][str(e + 1)] = round(tok_do / max(time.time() - t0, 1e-6), 1)\n            ln, dm = loss_nhom(m, tok, kiem, thiet_bi)\n            m.train()\n            v["loss"][str(e + 1)], v["diem"][str(e + 1)] = ln, dm\n            v["epoch_xong"], v["vi_tri_lo"] = e + 1, 0\n            if e + 1 == SO_EPOCH_B6:\n                m.save_pretrained(str(d / f"ep{e + 1}"))\n                ghi()\n                shutil.rmtree(moi_nhat, ignore_errors=True)    # xong việc thì bỏ, mỗi bản kèm opt.pt ~1,6 GB\n            else:\n                luu()\n            print(ten, "XONG epoch", e + 1, "diem loss", round(dm, 5), ln, "token/giay", v["token_moi_giay"][str(e + 1)], "gio da chay", round((time.time() - t_bat_dau) / 3600, 2), flush=True)\n        goc = m.unload()\n        del m, opt\n        torch.cuda.empty_cache()\n\n    def danh_gia(ten, v, nhanh, fold, thu_adapter, tep_sinh, ca_fold=True):\n        nonlocal goc\n        if v.get("hong") or ((v.get("loss_ca_fold") or not ca_fold) and v.get("cham")):\n            return\n        kiem_gio(ten)\n        m = PeftModel.from_pretrained(goc, str(thu_adapter))\n        if ca_fold and not v.get("loss_ca_fold"):\n            ca_fold = [x for x in ma[nhanh] if x["fold5"] == fold]\n            if cfg.get("so_mau_ca_fold"):\n                ca_fold = random.Random(cfg["seed"]).sample(ca_fold, cfg["so_mau_ca_fold"])\n            theo = {}\n            for x in ca_fold:\n                theo.setdefault(f"{x[\'huong\']}|{x[\'loai\']}|{x[\'category\']}", []).append(x)\n            t = time.time()\n            tung = {k: loss_tong(m, tok, a, 32, thiet_bi) for k, a in sorted(theo.items())}\n            nhom, dm = gop_loss(tung)\n            v["loss_ca_fold"] = {"so_mau": len(ca_fold), "diem_macro": dm, "theo_nhom": nhom, "theo_category": {k: {"loss": a / max(n, 1), "so_token": n} for k, (a, n) in tung.items()}}\n            ghi()\n            print(ten, "loss ca fold", round(dm, 5), nhom, "so mau", len(ca_fold), "giay", round(time.time() - t), flush=True)\n        if not v.get("cham"):\n            kiem_gio(ten)\n            mau, thieu = lay_mau_sinh(nhanh, fold)\n            v["so_id_sinh_thieu"] = thieu\n            if not tep_sinh.exists():\n                t = time.time()\n                du = sinh(m, tok, mau, thiet_bi)\n                v["giay_moi_mau"] = round((time.time() - t) / len(du), 3)\n                ghi_sinh(tep_sinh, du)\n            _, v["cham"] = cham_mot(nhanh, mau, doc_sinh(tep_sinh, mau), tap_da_thay(nhanh, fold), SO_EPOCH_B6, v.get("giay_moi_mau"))\n            ghi()\n            print(ten, "chrF", v["cham"]["chrf"]["chrf_macro_huong"], "so mau", len(mau), "thieu", thieu, "giay/mau", v.get("giay_moi_mau"), "gio da chay", round((time.time() - t_bat_dau) / 3600, 2), flush=True)\n        goc = m.unload()\n        del m\n        torch.cuda.empty_cache()\n\n    def kiem_nhanh_cuoi(ten, v, nhanh, thu_adapter):\n        """Bản cuối trên toàn pool: sinh thử vài mẫu (hướng x loại) CỦA CHÍNH TẬP HUẤN LUYỆN và chấm chrF, chỉ để bắt hỏng (đầu ra rỗng, lặp, chrF thấp bất thường).\n        Điểm này KHÔNG phải chất lượng trên dữ liệu mới; chất lượng dựa trên K-fold và lần đo test của adapter fold 0."""\n        nonlocal goc\n        if v.get("hong") or v.get("kiem_nhanh"):\n            return\n        kiem_gio(ten)\n        loai_sp = set(cfg.get("loai_title_sp") or []) if v.get("so_title_bi_loai") else set()\n        mau = lay_phan_tang([x for x in goc_mau[nhanh] if x["phan"] == 0 and not (x["loai"] == "title" and x["product_id"] in loai_sp)], cfg.get("so_sinh_kiem_nhanh", 25), cfg["seed"] + 7)\n        m = PeftModel.from_pretrained(goc, str(thu_adapter))\n        t = time.time()\n        du = sinh(m, tok, mau, thiet_bi)\n        giay = round((time.time() - t) / len(du), 3)\n        goc = m.unload()\n        del m\n        torch.cuda.empty_cache()\n        _, v["kiem_nhanh"] = cham_mot(nhanh, mau, du, None, SO_EPOCH_B6, giay)\n        v["kiem_nhanh"]["tren_du_lieu_huan_luyen"] = True\n        ghi()\n        print(ten, "KIEM NHANH (tren du lieu huan luyen) chrF", v["kiem_nhanh"]["chrf"]["chrf_macro_huong"], "so mau", len(mau), "sinh lap", v["kiem_nhanh"]["sinh_lap"], flush=True)\n\n    # Việc 1: LoRA attention so với LoRA mọi lớp cùng lr, rank, epoch 1, cụm fold 0, cùng mẫu sinh của B6. Khảo sát tỉ lệ không cần phần này\n    mau_a, tm_a = None, {}\n    if not cfg.get("viec"):\n        mau_a, _ = lay_mau_sinh("cum", 0)\n        (thu_ra / "attn").mkdir(exist_ok=True)\n    for ten in (() if cfg.get("viec") else CAP_ATTN):\n        tep_a = thu_ra / "attn" / f"{ten}_sinh.jsonl"\n        if ten not in st["attn"]:\n            kiem_gio(f"attn {ten}")\n            m = PeftModel.from_pretrained(goc, str(kho / ten / "ep1"))\n            t = time.time()\n            du = sinh(m, tok, mau_a, thiet_bi)\n            giay = round((time.time() - t) / len(du), 3)\n            ghi_sinh(tep_a, du)\n            goc = m.unload()\n            del m\n            torch.cuda.empty_cache()\n            _, st["attn"][ten] = cham_mot("cum", mau_a, du, None, 1, giay)\n            ghi()\n            print(ten, "epoch 1 chrF", st["attn"][ten]["chrf"]["chrf_macro_huong"], "giay/mau", giay, flush=True)\n        tm_a[ten], _ = cham_mot("cum", mau_a, doc_sinh(tep_a, mau_a), None, 1, None)\n    if not cfg.get("viec") and "so_cap" not in st["attn"]:\n        ids_a = [m["id"] for m in mau_a]\n        a, b = (tm_a[t].set_index("id").chrf[ids_a].tolist() for t in CAP_ATTN)\n        st["attn"]["so_cap"] = bootstrap_chenh(a, b, [f"{m[\'huong\']}|{m[\'loai\']}" for m in mau_a])\n        ghi()\n        print("ATTN - MOI LOP:", st["attn"]["so_cap"], flush=True)\n\n    # Việc 2 đến 5\n    ten_chot = CAU_HINH_CHOT["ten"]\n    for j in ds_viec:\n        ten, nhanh, fold = j["ten"], j["nhanh"], j["fold"]\n        v = st["viec"].setdefault(ten, {"nhanh": nhanh, "fold": fold, "cau_hinh": ten_chot, "epoch_xong": 0, "buoc": 0, "vi_tri_lo": 0, "loss": {}, "diem": {}, "token_moi_giay": {}, "loss_ca_fold": None, "cham": None})\n        if ten == "cum_f0":\n            b = st_b6["cau_hinh"][ten_chot]\n            v.update(nguon_adapter="b6", epoch_xong=b["epoch_xong"], loss=b["loss"], diem=b["diem"], token_moi_giay=b["token_moi_giay"], giay_moi_mau=st_b6["cham"][ten_chot]["giay_moi_mau"])\n            danh_gia(ten, v, nhanh, fold, kho / ten_chot / "ep2", kho / ten_chot / "sinh_ep2.jsonl")\n            if not cfg.get("rut_gon_sinh") and v.get("cham"):      # chấm lại bản dịch B6 phải ra đúng điểm B6\n                goc_b6 = st_b6["cham"][ten_chot]["chrf"]["chrf_macro_huong"]\n                v["khop_b6"] = abs(v["cham"]["chrf"]["chrf_macro_huong"] - goc_b6) < 1e-6\n                ghi()\n                if not v["khop_b6"]:\n                    raise RuntimeError(f"CHAM_LAI_KHONG_KHOP_B6:{v[\'cham\'][\'chrf\'][\'chrf_macro_huong\']}:{goc_b6}")\n            continue\n        huan_luyen(ten, v, nhanh, fold, j.get("p_title"), j.get("lr"), set(cfg.get("loai_title_sp") or []) if j.get("loai_title_co") else None)\n        if cfg.get("chrf_toi_thieu") is not None and v.get("hong"):\n            dung_bat_thuong(ten, "loss khong huu han")\n        if j.get("khong_danh_gia"):         # bản cuối trên toàn pool: không còn fold giữ lại; chỉ sinh thử vài mẫu để bắt hỏng\n            kiem_nhanh_cuoi(ten, v, nhanh, thu_ra / ten / f"ep{SO_EPOCH_B6}")\n            continue\n        danh_gia(ten, v, nhanh, fold, thu_ra / ten / f"ep{SO_EPOCH_B6}", thu_ra / ten / f"sinh_ep{SO_EPOCH_B6}.jsonl", j.get("ca_fold", True))\n        if cfg.get("chrf_toi_thieu") is not None and v.get("cham"):\n            ly_do = kiem_bat_thuong(v["cham"], cfg["chrf_toi_thieu"], cfg["lap_toi_da"])\n            if ly_do:\n                dung_bat_thuong(ten, ly_do)\n    st["xong"] = True\n    ghi()\n    bc["dat"] = not any(v.get("hong") for v in st["viec"].values())\n    print("B78 XONG", {t: (v["cham"] or {}).get("chrf", {}).get("chrf_macro_huong") for t, v in st["viec"].items()}, flush=True)\n\n\ndef chay_do_test(cfg, bc):\n    """Đo MỘT lần (B10, Nhật Anh duyệt 07/11/2026, Prompt 806): với mỗi nhánh (cụm, pair), sinh bản dịch của zero-shot (Qwen3-4B nguyên bản) và của adapter fold 0 ở ~11% title\n    trên cùng mẫu, trần sinh theo độ dài NGUỒN, rồi chấm bằng cham_diem. Không huấn luyện gì, không đổi cấu hình theo điểm.\n    cfg: nguon_do = "test" (đọc do_khoa_{nhanh}.jsonl* từ Dataset test riêng, kiểm không rò vào pool) hoặc "pool_fold" (CHỈ ĐỂ CHẠY KHÓI: lấy mẫu của một fold pool làm \'test\' giả,\n    không đụng test thật); mo_hinh_ft = {nhanh: tên việc trong kho lưu}; nhanh; gio_toi_da; so_mau_do (mỗi nhóm hướng x loại, chỉ khói).\n    Ra: do_test/do_test_trang_thai.json (điểm), do_test/sinh_{nhanh}_{mo_hinh}.jsonl (bản dịch), do_test/sinh_diem_{nhanh}_{mo_hinh}.jsonl (điểm từng mục, KHÔNG có tham chiếu).\n    Mỗi (nhánh, mô hình) tự bỏ qua nếu đã xong; hết giờ thì lưu rồi dừng có trật tự, phiên sau chạy tiếp."""\n    import gc\n    import hashlib\n    import shutil\n    import pandas as pd\n    import torch\n    import transformers\n    import peft\n    from peft import PeftModel\n    from transformers import AutoTokenizer\n    import cham_diem as D\n\n    thiet_bi = "cuda:0"\n    het_giay = cfg.get("gio_toi_da", 4.5) * 3600\n    thu_ra = RA / "do_test"\n    tep = thu_ra / "do_test_trang_thai.json"\n    if not tep.exists():\n        cu = sorted(VAO.rglob("do_test_trang_thai.json"))\n        if cu:\n            shutil.copytree(cu[0].parent, thu_ra)\n            print("chay tiep tu:", cu[0], flush=True)\n    thu_ra.mkdir(parents=True, exist_ok=True)\n    st = json.loads(tep.read_text(encoding="utf-8")) if tep.exists() else {"xong": False, "bat_thuong": None, "ket_qua": {}}\n    if st.get("nguon_do") not in (None, cfg["nguon_do"]):\n        raise RuntimeError(f"NOI_NHAM_NGUON_DO:{st.get(\'nguon_do\')}:{cfg[\'nguon_do\']}")          # không nối khói (mẫu pool) vào lần đo test thật hay ngược lại\n\n    def ghi():\n        tam = tep.with_suffix(".tmp")\n        tam.write_text(json.dumps(st, ensure_ascii=False, indent=1, default=str), encoding="utf-8")\n        tam.replace(tep)\n\n    bc["moi_truong"] = {"torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__, "gpu": torch.cuda.get_device_name(0)}\n    thu_mh = tim_mot("REVISION.txt").parent\n    bc["mo_hinh"] = thu_mh.joinpath("REVISION.txt").read_text().split()\n    tok = AutoTokenizer.from_pretrained(str(thu_mh))\n    if tok.pad_token_id is None:\n        tok.pad_token = tok.eos_token\n    tok.padding_side = "right"\n    kho_tt = tim_mot("tl_trang_thai.json")\n    kho = kho_tt.parent\n    mf_kho = json.loads(tim_mot("MANIFEST_TL.json").read_text(encoding="utf-8"))          # nằm ngoài thư mục tl/ của kho lưu; khóa là đường dẫn tương đối so với tl/\n    st["nguon_do"] = cfg["nguon_do"]\n    st["mo_hinh_ft"] = cfg["mo_hinh_ft"]\n    st["cach_sinh"] = {"giai_ma": "greedy", "tran": "min(2048, ceil(3,5 x token nguon) + 64)", "enable_thinking": False}\n    ghi()\n\n    for nhanh in cfg["nhanh"]:\n        pool = doc_mau(nhanh)\n        if cfg["nguon_do"] == "test":\n            ds = [p for p in sorted(VAO.rglob(f"do_khoa_{nhanh}.jsonl*")) if p.is_file()]\n            if len(ds) != 1:\n                raise RuntimeError(f"do_khoa_{nhanh}: thay {len(ds)} file")\n            with open(ds[0], "rb") as f:\n                nen = f.read(2) == b"\\x1f\\x8b"\n            mo = gzip.open(ds[0], "rt", encoding="utf-8") if nen else open(ds[0], encoding="utf-8")\n            with mo as f:\n                mau = [json.loads(x) for x in f]\n            n_sp, n_mau = kiem_ro_ri_test(mau, pool)\n            st.setdefault("mau_do", {})[nhanh] = {"so_sp": n_sp, "so_mau": n_mau, "so_phan_khac_0": sum(1 for m in mau if m.get("phan", 0) != 0),\n                                                   "sha256": hashlib.sha256(ds[0].read_bytes()).hexdigest()}\n        else:\n            mau = lay_phan_tang([m for m in pool if m["fold5"] == cfg.get("fold_do", 4) and m["phan"] == 0], cfg.get("so_mau_do", 3), cfg["seed"])\n            st.setdefault("mau_do", {})[nhanh] = {"so_mau": len(mau), "ghi_chu": "KHOI: mau cua fold pool, khong phai test"}\n        for m in mau:\n            for k in ("id", "product_id", "huong", "loai", "nguon", "dich", "messages", "category"):\n                if k not in m:\n                    raise RuntimeError(f"mau_do_thieu_truong:{nhanh}:{k}")\n        # "đã thấy" = dòng nguồn có trong các fold mà adapter fold 0 đã học (fold khác 0), cùng cách với phần fold\n        tap = D.xay_tap_da_thay(pd.DataFrame(pool)[["loai", "category", "nguon", "nhanh", "fold5"]], cfg.get("fold_adapter", 0), nhanh)\n        ten_ft = cfg["mo_hinh_ft"][nhanh]\n        duong_ad = kho / ten_ft / "ep2"\n        rel = f"{ten_ft}/ep2/adapter_model.safetensors"\n        h = hashlib.sha256()\n        with open(duong_ad / "adapter_model.safetensors", "rb") as a:\n            for k in iter(lambda: a.read(2**20), b""):\n                h.update(k)\n        if h.hexdigest() != mf_kho[rel]["sha256"]:\n            raise RuntimeError(f"ADAPTER_LECH_HASH:{rel}")\n        st.setdefault("adapter", {})[nhanh] = {"ten": ten_ft, "sha256": h.hexdigest()}\n        ghi()\n        for mh in ("zero_shot", "ft"):\n            khoa = f"{nhanh}/{mh}"\n            if khoa in st["ket_qua"]:\n                continue\n            if time.time() - cfg["_t_bat_dau"] > het_giay:\n                raise DungHetGio(khoa)\n            goc = nap_goc(thu_mh)\n            model = PeftModel.from_pretrained(goc, str(duong_ad)) if mh == "ft" else goc\n            t0 = time.time()\n            du = sinh(model, tok, mau, thiet_bi, tran_theo="nguon")\n            giay = (time.time() - t0) / max(1, len(mau))\n            (thu_ra / f"sinh_{nhanh}_{mh}.jsonl").write_text("\\n".join(json.dumps(r, ensure_ascii=False) for r in du) + "\\n", encoding="utf-8")\n            tm, th = D.cham(pd.DataFrame(mau), pd.DataFrame(du), nhanh=nhanh, tap_da_thay=tap)\n            pid = {m["id"]: m["product_id"] for m in mau}\n            cot = [c for c in tm.columns if not c.startswith("_")]\n            (thu_ra / f"sinh_diem_{nhanh}_{mh}.jsonl").write_text(\n                "\\n".join(json.dumps(D._sach({**{c: r[c] for c in cot}, "product_id": pid[r["id"]]}), ensure_ascii=False) for r in tm.to_dict("records")) + "\\n", encoding="utf-8")\n            st["ket_qua"][khoa] = D._sach({"so_mau": len(mau), "giay_moi_mau": round(giay, 3), "sinh_lap": int(tm.sinh_lap.sum()), "thieu_hyp": th["thieu_hyp"],\n                                           "ti_le_cat_loi_dan": float(tm.da_cat_loi_dan.mean()), "chrf": th["diem_chinh"], "theo_huong": th["theo_huong"],\n                                           "theo_nhom": th["theo_nhom"], "cat_loi_dan": th["cat_loi_dan"], "dong_da_thay_moi": th.get("dong_da_thay_moi")})\n            ghi()\n            print(f"{khoa} XONG chrF {th[\'diem_chinh\'][\'chrf_macro_huong\']:.2f} so mau {len(mau)} giay/mau {giay:.3f}", flush=True)\n            del model, goc\n            gc.collect()\n            torch.cuda.empty_cache()\n    st["xong"] = all(f"{n}/{m}" in st["ket_qua"] for n in cfg["nhanh"] for m in ("zero_shot", "ft"))\n    ghi()\n    bc["dat"] = st["xong"]\n    print("DO_TEST XONG" if st["xong"] else "DO_TEST CHUA XONG", flush=True)\n\n\ndef thu_b5(cfg, bc):\n    import torch\n    import transformers\n    from peft import LoraConfig, PeftModel, get_peft_model\n    from transformers import AutoTokenizer\n    import peft\n\n    torch.manual_seed(cfg["seed"])\n    random.seed(cfg["seed"])\n    thiet_bi = "cuda:0"\n    bc["moi_truong"] = {"python": sys.version.split()[0], "torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__,\n                        "gpu": torch.cuda.get_device_name(0), "vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1),\n                        "cuda_khả_dụng": torch.cuda.is_available()}\n    print(bc["moi_truong"], flush=True)\n    thu_mh = tim_mot("REVISION.txt").parent\n    bc["mo_hinh"] = thu_mh.joinpath("REVISION.txt").read_text().split()\n    tok = AutoTokenizer.from_pretrained(str(thu_mh))\n    if tok.pad_token_id is None:\n        tok.pad_token = tok.eos_token\n    tok.padding_side = "right"\n\n    goc_mau = doc_mau(cfg["nhanh"])\n    bc["so_mau_pool"] = len(goc_mau)\n    ma, bo = xay_mau(goc_mau, tok, cfg["max_len"])\n    bc["so_mau_vuot_max_len_bi_bo"] = bo\n    bc["so_token_toan_bo_1_epoch"] = sum(len(m["ids"]) for m in ma)\n    v0 = ma[0]\n    print("MAU MA HOA:\\n", tok.decode(v0["ids"])[:600], "\\nPHAN TINH LOSS:\\n", tok.decode([x for x in v0["labels"] if x != -100])[:300], flush=True)\n    train = [m for m in ma if m["fold5"] != cfg["fold_kiem"]]\n    kiem_all = [m for m in ma if m["fold5"] == cfg["fold_kiem"]]\n    bc["so_mau_train"], bc["so_mau_kiem"] = len(train), len(kiem_all)\n    if set(m["id"] for m in train) & set(m["id"] for m in kiem_all):\n        raise RuntimeError("giao_train_kiem")\n    rng = random.Random(cfg["seed"])\n    kiem = rng.sample(kiem_all, cfg["so_mau_kiem"])\n\n    t = time.time()\n    model = nap_goc(thu_mh)\n    bc["giay_nap_mo_hinh"] = round(time.time() - t, 1)\n    model = get_peft_model(model, LoraConfig(r=cfg["rank"], lora_alpha=2 * cfg["rank"], lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM"))\n    # Đo 05/11/2026: không checkpoint + lô 4096 token nhanh nhất (5.873 token/giây, 52 GB); 8192 token không checkpoint tràn 95 GB\n    if cfg.get("gc", False):\n        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n        model.enable_input_require_grads()\n    model.config.use_cache = False\n    tham_so = [p for p in model.parameters() if p.requires_grad]\n    bc["so_tham_so_lora"] = sum(p.numel() for p in tham_so)\n    print("tham so LoRA:", bc["so_tham_so_lora"], flush=True)\n\n    bc["loss_kiem_truoc"] = loss_kiem(model, tok, kiem, cfg["micro_bs"] * 2, thiet_bi)\n    print("loss kiem truoc:", bc["loss_kiem_truoc"], flush=True)\n\n    opt = torch.optim.AdamW(tham_so, lr=cfg["lr"], weight_decay=0.0)\n    ngan_sach = cfg.get("ngan_sach_token", 4096)\n    lo = chia_lo(train, cfg["micro_bs"], cfg["seed"], ngan_sach)\n    bc["ngan_sach_token"] = ngan_sach\n    co_batch = cfg["micro_bs"] * cfg["tich_luy"]        # batch hiệu dụng tính theo số mẫu\n    tong_buoc = cfg["so_buoc"]\n    model.train()\n    nhat_ky, tok_dem, tok_do, t_do = [], 0, 0, None\n    torch.cuda.reset_peak_memory_stats()\n    t0 = time.time()\n    it = iter(lo)\n    for buoc in range(tong_buoc):\n        if buoc == 5:                    # bỏ 5 bước đầu khỏi phép đo tốc độ\n            t_do, tok_do = time.time(), 0\n        tong, da_co = 0.0, 0\n        while da_co < co_batch:          # gom đủ batch hiệu dụng; lô dài có ít mẫu hơn nên số micro-batch thay đổi\n            k = next(it)\n            ids, lab, mask = lo_tensor(k, tok, thiet_bi)\n            out = model(input_ids=ids, attention_mask=mask, labels=lab)\n            (out.loss * len(k) / co_batch).backward()\n            tong += out.loss.item() * len(k) / co_batch\n            da_co += len(k)\n            n = int(mask.sum().item())\n            tok_dem += n\n            tok_do += n\n        for g in opt.param_groups:\n            g["lr"] = cfg["lr"] * lich_lr(buoc, tong_buoc)\n        torch.nn.utils.clip_grad_norm_(tham_so, 1.0)\n        opt.step()\n        opt.zero_grad(set_to_none=True)\n        if buoc % 10 == 0 or buoc == tong_buoc - 1:\n            nhat_ky.append({"buoc": buoc, "loss": round(tong, 4), "giay": round(time.time() - t0, 1)})\n            print(nhat_ky[-1], flush=True)\n    bc["nhat_ky_loss"] = nhat_ky\n    bc["token_moi_giay"] = round(tok_do / max(time.time() - t_do, 1e-6), 1)\n    bc["tong_token_da_chay"] = tok_dem\n    bc["vram_dinh_gb"] = round(torch.cuda.max_memory_allocated() / 2**30, 2)\n    print("token/giay:", bc["token_moi_giay"], "VRAM dinh GB:", bc["vram_dinh_gb"], flush=True)\n\n    bc["loss_kiem_sau"] = loss_kiem(model, tok, kiem, cfg["micro_bs"] * 2, thiet_bi)\n    print("loss kiem sau:", bc["loss_kiem_sau"], flush=True)\n    bc["loss_giam"] = bc["loss_kiem_sau"] < bc["loss_kiem_truoc"]\n\n    ck = RA / "ck_b5"\n    model.save_pretrained(str(ck))\n    bc["ck_mb"] = round(sum(p.stat().st_size for p in ck.rglob("*") if p.is_file()) / 2**20, 1)\n    del model, opt\n    torch.cuda.empty_cache()\n    model = PeftModel.from_pretrained(nap_goc(thu_mh), str(ck))\n    bc["loss_kiem_sau_nap_lai"] = loss_kiem(model, tok, kiem, cfg["micro_bs"] * 2, thiet_bi)\n    bc["nap_lai_khop"] = abs(bc["loss_kiem_sau_nap_lai"] - bc["loss_kiem_sau"]) < 1e-3\n    print("loss sau nap lai:", bc["loss_kiem_sau_nap_lai"], "khop:", bc["nap_lai_khop"], flush=True)\n\n    # Sinh thử trên fold kiểm (không dùng test): chia đều theo hướng x loại\n    chon, n_moi = [], max(1, cfg["so_mau_sinh"] // 4)\n    id_co = {m["id"] for m in kiem}\n    for h in ("zh->vi", "vi->zh"):\n        for l in ("title", "mo_ta"):\n            ung = [m for m in goc_mau if m["fold5"] == cfg["fold_kiem"] and m["huong"] == h and m["loai"] == l and m["phan"] == 0 and m["id"] in id_co]\n            chon += rng.sample(ung, min(n_moi, len(ung)))\n    t = time.time()\n    du_ft = sinh(model, tok, chon, thiet_bi)\n    bc["giay_sinh_moi_mau_ft"] = round((time.time() - t) / max(1, len(chon)), 2)\n    t = time.time()\n    with model.disable_adapter():\n        du_zs = sinh(model, tok, chon, thiet_bi)\n    bc["giay_sinh_moi_mau_zs"] = round((time.time() - t) / max(1, len(chon)), 2)\n\n    import pandas as pd\n    import cham_diem as D\n    tc = pd.DataFrame(chon)\n    for ten, du in (("fine_tune", du_ft), ("zero_shot", du_zs)):\n        (RA / f"sinh_{ten}.jsonl").write_text("\\n".join(json.dumps(r, ensure_ascii=False) for r in du) + "\\n", encoding="utf-8")\n        tm, th = D.cham(tc, pd.DataFrame(du), nhanh=cfg["nhanh"])\n        bc[f"cham_{ten}"] = th.get("diem_chinh")\n        bc[f"cham_{ten}_ti_le_cat_loi_dan"] = float(tm.da_cat_loi_dan.mean())\n    bc["vi_du_sinh"] = [{"id": m["id"], "huong": m["huong"], "loai": m["loai"], "nguon": m["nguon"][:200], "dich": m["dich"][:200],\n                         "ft": a["hyp_tho"][:200], "zs": b["hyp_tho"][:300]} for m, a, b in zip(chon, du_ft, du_zs)]\n\n\ndef do_toc_do(cfg, bc):\n    """B5: so token/giây và VRAM giữa các cách chia lô (kể cả lô dài nhất để biết có tràn không). Chỉ đo tốc độ, không lưu mô hình."""\n    import torch\n    from peft import LoraConfig, get_peft_model\n    from transformers import AutoTokenizer\n\n    thiet_bi = "cuda:0"\n    thu_mh = tim_mot("REVISION.txt").parent\n    tok = AutoTokenizer.from_pretrained(str(thu_mh))\n    if tok.pad_token_id is None:\n        tok.pad_token = tok.eos_token\n    tok.padding_side = "right"\n    ma, _ = xay_mau(doc_mau(cfg["nhanh"]), tok, cfg["max_len"])\n    train = [m for m in ma if m["fold5"] != cfg["fold_kiem"]]\n    model = get_peft_model(nap_goc(thu_mh), LoraConfig(r=cfg["rank"], lora_alpha=2 * cfg["rank"], lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM"))\n    model.enable_input_require_grads()\n    model.config.use_cache = False\n    model.train()\n    tham_so = [p for p in model.parameters() if p.requires_grad]\n    opt = torch.optim.AdamW(tham_so, lr=cfg["lr"], weight_decay=0.0)\n    dai_nhat = sorted(train, key=lambda m: -len(m["ids"]))\n    bc["do_toc_do"] = []\n    for c in cfg["cac_cau_hinh"]:\n        kq = dict(c)\n        if c["gc"]:\n            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n        else:\n            model.gradient_checkpointing_disable()\n        torch.cuda.empty_cache()\n        torch.cuda.reset_peak_memory_stats()\n        try:\n            xau = cat_ngan_sach(dai_nhat[:64], c["micro_bs"], c["ngan_sach"])[0]     # lô tệ nhất\n            ids, lab, mask = lo_tensor(xau, tok, thiet_bi)\n            model(input_ids=ids, attention_mask=mask, labels=lab).loss.backward()\n            opt.zero_grad(set_to_none=True)\n            kq["vram_lo_dai_nhat_gb"] = round(torch.cuda.max_memory_allocated() / 2**30, 1)\n            it = iter(chia_lo(train, c["micro_bs"], cfg["seed"], c["ngan_sach"]))\n            dem, t0 = 0, None\n            for buoc in range(cfg["so_buoc_do"]):\n                if buoc == 3:\n                    dem, t0 = 0, time.time()\n                da_co = 0\n                while da_co < 32:\n                    k = next(it)\n                    ids, lab, mask = lo_tensor(k, tok, thiet_bi)\n                    (model(input_ids=ids, attention_mask=mask, labels=lab).loss * len(k) / 32).backward()\n                    da_co += len(k)\n                    dem += int(mask.sum().item())\n                opt.step()\n                opt.zero_grad(set_to_none=True)\n            kq["token_moi_giay"] = round(dem / (time.time() - t0), 1)\n            kq["vram_dinh_gb"] = round(torch.cuda.max_memory_allocated() / 2**30, 1)\n        except torch.OutOfMemoryError:\n            kq["tran_bo_nho"] = True\n            opt.zero_grad(set_to_none=True)\n            torch.cuda.empty_cache()\n        bc["do_toc_do"].append(kq)\n        print(kq, flush=True)\n    bc["dat"] = True\n\n\ndef do_nguyen_nhan(cfg, bc):\n    """Tìm chỗ chậm trong một bước huấn luyện: thời gian từng phần trên cùng một bộ lô cố định, GPU có chạy hết công suất không, và vài biến thể."""\n    import os\n    import threading\n    import torch\n    import torch.nn.functional as F\n    from peft import LoraConfig, get_peft_model\n    from transformers import AutoTokenizer\n\n    thiet_bi = "cuda:0"\n    thu_mh = tim_mot("REVISION.txt").parent\n    tok = AutoTokenizer.from_pretrained(str(thu_mh))\n    if tok.pad_token_id is None:\n        tok.pad_token = tok.eos_token\n    tok.padding_side = "right"\n    ma, _ = xay_mau(doc_mau(cfg["nhanh"]), tok, cfg["max_len"])\n    train = [m for m in ma if m["fold5"] != cfg["fold_kiem"]]\n    lo = chia_lo(train, cfg["micro_bs"], cfg["seed"], cfg["ngan_sach_token"])[:cfg["so_lo_do"]]\n    goc = nap_goc(thu_mh)\n    c = goc.config\n    tong_tok = sum(len(m["ids"]) for k in lo for m in k)\n    o_dem = sum(len(k) * max(len(m["ids"]) for m in k) for k in lo)\n    tinh_loss = sum(sum(x != -100 for x in m["labels"]) for k in lo for m in k)\n    bc["mo_hinh_cau_truc"] = {"so_lop": c.num_hidden_layers, "hidden": c.hidden_size, "tu_vung": c.vocab_size, "attn": getattr(c, "_attn_implementation", None),\n                              "so_cpu": os.cpu_count(), "dtype": str(next(goc.parameters()).dtype)}\n    bc["lo_do"] = {"so_lo": len(lo), "token_that": tong_tok, "o_tinh_gom_dem": o_dem, "ti_le_dem": round(1 - tong_tok / o_dem, 3),\n                   "token_tinh_loss": tinh_loss, "ti_le_token_tinh_loss": round(tinh_loss / tong_tok, 3)}\n    print(bc["mo_hinh_cau_truc"], bc["lo_do"], flush=True)\n\n    def dong_ho():\n        torch.cuda.synchronize()\n        return time.perf_counter()\n\n    def forward(model, ids, lab, mask, chi_dich):\n        if not chi_dich:\n            return model(input_ids=ids, attention_mask=mask, labels=lab).loss\n        trong = model.base_model.model                       # chỉ tính lm_head ở vị trí có loss, tránh ma trận logits toàn bộ\n        h = trong.model(input_ids=ids, attention_mask=mask).last_hidden_state[:, :-1]\n        nhan = lab[:, 1:]\n        chon = nhan != -100\n        return F.cross_entropy(trong.lm_head(h[chon]).float(), nhan[chon])\n\n    def do_bien_the(model, ten, chi_dich=False, lay_gpu=False):\n        tham_so = [p for p in model.parameters() if p.requires_grad]\n        opt = torch.optim.AdamW(tham_so, lr=1e-4, weight_decay=0.0)\n        model.train()\n        torch.cuda.empty_cache()\n        torch.cuda.reset_peak_memory_stats()\n        tg = {"tensor": 0.0, "forward": 0.0, "backward": 0.0, "optimizer": 0.0}\n        mau_gpu, dung = [], threading.Event()\n\n        def lay():\n            while not dung.is_set():\n                r = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,clocks.sm,clocks.max.sm,power.draw,power.limit,temperature.gpu", "--format=csv,noheader,nounits"], capture_output=True, text=True)\n                mau_gpu.append(r.stdout.strip())\n                time.sleep(1)\n\n        if lay_gpu:\n            threading.Thread(target=lay, daemon=True).start()\n        dem = 0\n        for i, k in enumerate(lo):\n            t = dong_ho()\n            ids, lab, mask = lo_tensor(k, tok, thiet_bi)\n            t1 = dong_ho()\n            loss = forward(model, ids, lab, mask, chi_dich)\n            t2 = dong_ho()\n            loss.backward()\n            t3 = dong_ho()\n            torch.nn.utils.clip_grad_norm_(tham_so, 1.0)\n            opt.step()\n            opt.zero_grad(set_to_none=True)\n            t4 = dong_ho()\n            if i >= 3:                    # bỏ vài lô đầu\n                tg["tensor"] += t1 - t\n                tg["forward"] += t2 - t1\n                tg["backward"] += t3 - t2\n                tg["optimizer"] += t4 - t3\n                dem += int(mask.sum().item())\n        dung.set()\n        tong = sum(tg.values())\n        kq = {"ten": ten, "token_moi_giay": round(dem / tong, 1), "vram_dinh_gb": round(torch.cuda.max_memory_allocated() / 2**30, 1),\n              "ti_trong_%": {a: round(100 * b / tong, 1) for a, b in tg.items()}, "so_tham_so_lora": sum(p.numel() for p in tham_so)}\n        if mau_gpu:\n            kq["nvidia_smi_mau"] = mau_gpu[len(mau_gpu) // 2:len(mau_gpu) // 2 + 4]\n        bc["bien_the"].append(kq)\n        print(kq, flush=True)\n        del opt\n\n    # Mốc: chỉ chạy suy luận mô hình gốc, không LoRA, không gradient\n    goc.eval()\n    t = dong_ho()\n    dem = 0\n    with torch.no_grad():\n        for k in lo:\n            ids, lab, mask = lo_tensor(k, tok, thiet_bi)\n            goc(input_ids=ids, attention_mask=mask)\n            dem += int(mask.sum().item())\n    bc["suy_luan_goc_token_moi_giay"] = round(dem / (dong_ho() - t), 1)\n    print("suy luan goc token/giay:", bc["suy_luan_goc_token_moi_giay"], flush=True)\n    bc["bien_the"] = []\n\n    def lora(model, r=16, dropout=0.05, muc_tieu="all-linear", **kw):\n        return get_peft_model(model, LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=dropout, target_modules=muc_tieu, task_type="CAUSAL_LM"), **kw)\n\n    for ten, args, chi_dich, lay_gpu in [\n            ("hien_tai_all_linear_r16", {}, False, True),\n            ("khong_dropout", {"dropout": 0.0}, False, False),\n            ("chi_attention_qkvo", {"muc_tieu": ["q_proj", "k_proj", "v_proj", "o_proj"]}, False, False),\n            ("r64", {"r": 64}, False, False),\n            ("lm_head_chi_vi_tri_dich", {}, True, False),\n            ("lora_cung_bf16", {"autocast_adapter_dtype": False}, False, False)]:\n        m = lora(goc, **args)\n        try:\n            do_bien_the(m, ten, chi_dich, lay_gpu)\n        except Exception:\n            bc["bien_the"].append({"ten": ten, "loi": traceback.format_exc()[-500:]})\n            print(bc["bien_the"][-1], flush=True)\n        goc = m.unload()\n        del m\n    bc["dat"] = True\n\n\ndef main(cfg):\n    import os\n    os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")\n    bc = {"cau_hinh": cfg, "dat": False}\n    t = time.time()\n    try:\n        if cfg["che_do"] == "do_test" and cfg.get("nguon_do") == "test":\n            kiem_khoa_test_do()\n            bc["khoa_test"] = "chi_co_do_khoa_dung_ten"          # lần đo test thật: Dataset test riêng được gắn; kiem_ro_ri_test kiểm không rò vào pool\n        else:\n            kiem_khoa_test()\n            bc["khoa_test"] = "khong_co_file_test_trong_input"\n        cai_thu_vien()\n        if cfg["che_do"] == "do_test":\n            cfg["_t_bat_dau"] = t\n            try:\n                chay_do_test(cfg, bc)\n            except DungHetGio as e:\n                bc["dung_het_gio"] = str(e)\n                print("KET THUC PHIEN CO TRAT TU, dung o", e, flush=True)\n        elif cfg["che_do"] in ("b6", "b78"):\n            cfg["_t_bat_dau"] = t\n            try:\n                (chay_b6 if cfg["che_do"] == "b6" else chay_b78)(cfg, bc)\n            except DungHetGio as e:\n                bc["dung_het_gio"] = str(e)            # kết thúc phiên bình thường để Kaggle lưu đầu ra; phiên sau chạy tiếp\n                print("KET THUC PHIEN CO TRAT TU, dung o", e, flush=True)\n        elif cfg["che_do"] == "do_toc_do":\n            do_toc_do(cfg, bc)\n        elif cfg["che_do"] == "do_nguyen_nhan":\n            do_nguyen_nhan(cfg, bc)\n        elif cfg["che_do"] == "thu_b5":\n            thu_b5(cfg, bc)\n            bc["dat"] = bool(bc.get("loss_giam") and bc.get("nap_lai_khop"))\n        else:\n            raise RuntimeError(f"che_do_chua_ho_tro:{cfg[\'che_do\']}")\n    except Exception:\n        bc["loi"] = traceback.format_exc()\n        print(bc["loi"], flush=True)\n    bc["tong_giay"] = round(time.time() - t, 1)\n    ten_bc = f"bao_cao_{cfg.get(\'ten_trang_thai\', cfg[\'che_do\'])}.json"\n    (RA / ten_bc).write_text(json.dumps(bc, ensure_ascii=False, indent=2, default=str), encoding="utf-8")\n    print("KET QUA:", json.dumps({k: v for k, v in bc.items() if k not in ("vi_du_sinh", "cau_hinh", "nhat_ky_loss", "b6", "b78")}, ensure_ascii=False, default=str)[:3000], flush=True)\n    if "loi" in bc:\n        raise RuntimeError(f"loi, xem {ten_bc}")\n', encoding='utf-8')
Path('kaggle_cheo.py').write_text('# -*- coding: utf-8 -*-\n"""Thử chéo định dạng pair và cụm (12/11/2026, Nhật Anh duyệt, Prompt 956). Chạy trên Kaggle, tắt Internet, KHÔNG có dữ liệu test.\n\nCâu hỏi: dịch mô tả sản phẩm zh->vi, đưa nguồn theo dạng pair (mỗi thuộc tính một dòng) hay dạng cụm (nối bằng "; "), và dùng adapter nào?\nMỗi sản phẩm của tập kiểm (fold 0, chỉ mô tả zh->vi) được dựng ĐÚNG MỘT nội dung nguồn ở cả hai dạng, rồi sinh bản dịch cho đủ 3 x 2 tổ hợp:\n  adapter in {cum_f0_p11, pair_f0_p11, zero_shot (không adapter)} x dạng đầu vào in {cum, pair}.\nHai adapter fold 0 được huấn luyện lại ở tài khoản 2 (chay_cheo_huan_luyen.py), chưa từng học mẫu của fold 0; không dùng adapter bản cuối vì nó đã học cả mẫu kiểm.\nChỉ SINH và lưu bản dịch; chấm điểm làm trên máy (cham_cheo.py), không tốn GPU.\n\nCấu hình (cfg): so_toi_da (số sản phẩm, mặc định 600), seed (mặc định 43 = seed 42 + 1 như chon_mau_sinh), gio_toi_da.\nRa: cheo/eval_items.jsonl (nguồn, tham chiếu, lời dẫn từng dạng), cheo/sinh_{adapter}_{dang}.jsonl, cheo/cheo_trang_thai.json, bao_cao_cheo.json.\n"""\nimport gc\nimport hashlib\nimport json\nimport random\nimport time\nimport traceback\nfrom pathlib import Path\n\nimport kaggle_train as K\n\nVAO = Path("/kaggle/input")\nRA = Path("/kaggle/working")\nHUONG = "zh->vi"\nTACH = {"cum": "; ", "pair": "\\n"}\nDANG = ("pair", "cum")\nADAPTER = ("zero_shot", "cum_f0_p11", "pair_f0_p11")\n\n# Chép đúng khuôn lời dẫn mô tả của dung_mau.py (nơi mọi mẫu huấn luyện được dựng). Phép thử dưới đây (kiem_loi_dan) bắt buộc lời dẫn dựng lại\n# khớp từng chữ với messages[0] của mẫu pool thật, nên nếu khuôn lệch thì chạy dừng ngay, không sinh bản dịch sai khuôn.\n_MO_TA_DAU = "Translate this product\'s attribute list from {SRC} to {TGT}. "\n_MO_TA_CUOI = "\\nProduct title (context only, do not translate): {TITLE}\\n\\n{NGUON}"\nCHI_DAN_MO_TA = {\n    "pair": _MO_TA_DAU + \'Each line is one attribute written as "name: value". \'\n            "Keep the same number and order of lines, and output only the translation." + _MO_TA_CUOI,\n    "cum": _MO_TA_DAU + \'The attributes are separated by "; " and each is written as "name: value". \'\n           \'Keep the same number and order of attributes, separated by "; ", and output only the translation.\' + _MO_TA_CUOI,\n}\n\n\ndef loi_dan(dang, nguon, title):\n    return CHI_DAN_MO_TA[dang].format(SRC="Chinese", TGT="Vietnamese", TITLE=title, NGUON=nguon)\n\n\ndef chon_eval(pool_cum, pool_pair, so_toi_da=600, seed=43, so_co_dinh=250):\n    """Sản phẩm kiểm: fold 0, mô tả nguyên (phan == 0), zh->vi, có ở cả hai nhánh, và nội dung NGUỒN hai nhánh giống hệt (nguồn cụm == các dòng nguồn pair nối bằng "; ").\n    Nhóm "cố định" = đúng các mẫu mô tả zh->vi mà mọi lần chạy huấn luyện đã sinh và chấm (chon_mau_sinh, 250 mỗi nhóm, seed 43), để đối chiếu lại điểm cũ.\n    Phần còn lại lấy ngẫu nhiên (seed cố định) tới so_toi_da sản phẩm. Trả (danh sách mục, thống kê)."""\n    cum = {m["id"]: m for m in pool_cum}\n    pair = {m["id"]: m for m in pool_pair}\n    ung = []\n    bo = {"khong_phai_zh_vi_fold0_nguyen": 0, "thieu_nhanh_hoac_title": 0, "nguon_hai_nhanh_khac_nhau": 0}\n    for id_ in sorted(cum):\n        mc = cum[id_]\n        if mc["loai"] != "mo_ta" or mc["fold5"] != 0 or mc["phan"] != 0 or mc["huong"] != HUONG:\n            bo["khong_phai_zh_vi_fold0_nguyen"] += 1\n            continue\n        mp, tit = pair.get(id_), cum.get(f"{mc[\'product_id\']}_title")\n        if mp is None or tit is None:\n            bo["thieu_nhanh_hoac_title"] += 1\n            continue\n        dong = mp["nguon"].split(TACH["pair"])\n        if TACH["cum"].join(dong) != mc["nguon"] or len(mp["dich"].split(TACH["pair"])) != len(dong):\n            bo["nguon_hai_nhanh_khac_nhau"] += 1\n            continue\n        ung.append({"id": id_, "product_id": mc["product_id"], "category": mc.get("category", ""), "title": tit["nguon"], "so_dong": len(dong),\n                    "nguon": {"pair": mp["nguon"], "cum": mc["nguon"]}, "ref": {"pair": mp["dich"], "cum": mc["dich"]},\n                    "loi_dan_goc": {"pair": mp["messages"][0]["content"], "cum": mc["messages"][0]["content"]}})\n    cd = {m["id"] for m in K.lay_phan_tang([m for m in pool_cum if m["fold5"] == 0 and m["phan"] == 0], so_co_dinh, seed)\n          if m["huong"] == HUONG and m["loai"] == "mo_ta"}\n    for x in ung:\n        x["co_dinh"] = x["id"] in cd\n    co_dinh = [x for x in ung if x["co_dinh"]]\n    con = [x for x in ung if not x["co_dinh"]]\n    random.Random(seed).shuffle(con)\n    ra = (co_dinh + con)[:so_toi_da]\n    ra.sort(key=lambda x: x["id"])\n    tk = {"so_ung_vien": len(ung), "so_chon": len(ra), "so_co_dinh_trong_chon": sum(x["co_dinh"] for x in ra), "so_co_dinh_goc": len(cd),\n          "so_co_dinh_bi_loai_do_nguon_khac": len(cd) - len(co_dinh), "bo": bo}\n    return ra, tk\n\n\ndef kiem_loi_dan(items):\n    """Lời dẫn dựng lại phải khớp từng chữ với lời dẫn đã dùng để huấn luyện (messages[0] của mẫu pool), ở cả hai dạng."""\n    sai = [(x["id"], d) for x in items for d in DANG if loi_dan(d, x["nguon"][d], x["title"]) != x["loi_dan_goc"][d]]\n    if sai:\n        raise RuntimeError(f"LOI_DAN_LECH_KHUON:{len(sai)}:{sai[:3]}")\n\n\ndef sha256(duong_dan):\n    h = hashlib.sha256()\n    with open(duong_dan, "rb") as f:\n        for k in iter(lambda: f.read(2 ** 20), b""):\n            h.update(k)\n    return h.hexdigest()\n\n\ndef tim_adapter(ten):\n    ds = sorted(p for p in VAO.rglob(f"{ten}/ep2/adapter_model.safetensors") if p.is_file())\n    if len(ds) != 1:\n        raise RuntimeError(f"adapter_{ten}: thay {len(ds)}: {[str(p) for p in ds]}")\n    cf = json.loads((ds[0].parent / "adapter_config.json").read_text(encoding="utf-8"))\n    if cf.get("r") != 64:\n        raise RuntimeError(f"adapter_{ten}: rank {cf.get(\'r\')} != 64")\n    return ds[0].parent\n\n\ndef doc_dong_json(p):\n    return [json.loads(x) for x in Path(p).read_text(encoding="utf-8").splitlines() if x.strip()]\n\n\ndef ghi_dong_json(p, rows):\n    tam = Path(p).with_suffix(".tmp")\n    tam.write_text("\\n".join(json.dumps(r, ensure_ascii=False) for r in rows) + "\\n", encoding="utf-8")\n    tam.replace(p)\n\n\ndef chay(cfg, bc):\n    import torch\n    import transformers\n    import peft\n    from peft import PeftModel\n    from transformers import AutoTokenizer\n\n    thiet_bi = "cuda:0"\n    t_bat_dau = time.time()\n    het_giay = cfg.get("gio_toi_da", 2.0) * 3600\n    thu_ra = RA / "cheo"\n    thu_ra.mkdir(parents=True, exist_ok=True)\n    tep = thu_ra / "cheo_trang_thai.json"\n    st = {"xong": False, "da_sinh": {}, "cau_hinh": {k: v for k, v in cfg.items() if not k.startswith("_")}}\n\n    def ghi():\n        tam = tep.with_suffix(".tmp")\n        tam.write_text(json.dumps(st, ensure_ascii=False, indent=1, default=str), encoding="utf-8")\n        tam.replace(tep)\n\n    bc["moi_truong"] = {"torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__, "gpu": torch.cuda.get_device_name(0)}\n    thu_mh = K.tim_mot("REVISION.txt").parent\n    bc["mo_hinh"] = thu_mh.joinpath("REVISION.txt").read_text().split()\n    tok = AutoTokenizer.from_pretrained(str(thu_mh))\n    if tok.pad_token_id is None:\n        tok.pad_token = tok.eos_token\n    tok.padding_side = "right"\n\n    pool_cum, pool_pair = K.doc_mau("cum"), K.doc_mau("pair")\n    items, tk = chon_eval(pool_cum, pool_pair, int(cfg.get("so_toi_da", 600)), int(cfg.get("seed", 43)))\n    if len(items) < 50:\n        raise RuntimeError(f"QUA_IT_MAU:{len(items)}:{tk}")\n    kiem_loi_dan(items)                      # lời dẫn dựng lại khớp từng chữ với mẫu huấn luyện thật\n    st["mau"] = tk\n    ghi_dong_json(thu_ra / "eval_items.jsonl", [{k: v for k, v in x.items() if k != "loi_dan_goc"} for x in items])\n    print("mau kiem:", json.dumps(tk, ensure_ascii=False), flush=True)\n    ghi()\n    duong = {t: tim_adapter(t) for t in ADAPTER if t != "zero_shot"}\n    st["adapter"] = {t: {"duong_dan": str(p), "sha256": sha256(p / "adapter_model.safetensors")} for t, p in duong.items()}\n    ghi()\n\n    goc = K.nap_goc(thu_mh)\n    for ten in ADAPTER:\n        con = [d for d in DANG if not (thu_ra / f"sinh_{ten}_{d}.jsonl").exists()]\n        if not con:\n            continue\n        if time.time() - t_bat_dau > het_giay:\n            st["dung_het_gio"] = ten\n            ghi()\n            return\n        m = goc if ten == "zero_shot" else PeftModel.from_pretrained(goc, str(duong[ten]))\n        for d in con:\n            mau = [{"id": x["id"], "messages": [{"role": "user", "content": loi_dan(d, x["nguon"][d], x["title"])}], "nguon": x["nguon"][d], "dich": x["ref"][d]} for x in items]\n            t0 = time.time()\n            du = K.sinh(m, tok, mau, thiet_bi, tran_theo="nguon")\n            giay = round((time.time() - t0) / len(du), 3)\n            ghi_dong_json(thu_ra / f"sinh_{ten}_{d}.jsonl", [{"id": r["id"], "hyp": r["hyp"]} for r in du])\n            st["da_sinh"][f"{ten}/{d}"] = {"so_mau": len(du), "giay_moi_mau": giay}\n            ghi()\n            print(f"{ten}/{d} XONG {len(du)} mau, {giay} giay/mau, gio da chay {round((time.time() - t_bat_dau) / 3600, 2)}", flush=True)\n        if ten != "zero_shot":\n            goc = m.unload()\n            del m\n        gc.collect()\n        torch.cuda.empty_cache()\n    st["xong"] = all(f"{t}/{d}" in st["da_sinh"] or (thu_ra / f"sinh_{t}_{d}.jsonl").exists() for t in ADAPTER for d in DANG)\n    ghi()\n\n\ndef main(cfg):\n    import os\n    os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")\n    bc = {"cau_hinh": cfg, "dat": False}\n    t = time.time()\n    try:\n        K.kiem_khoa_test()\n        bc["khoa_test"] = "khong_co_file_test_trong_input"\n        K.cai_thu_vien()\n        chay(cfg, bc)\n        bc["dat"] = True\n    except Exception:\n        bc["loi"] = traceback.format_exc()\n        print(bc["loi"], flush=True)\n    bc["tong_giay"] = round(time.time() - t, 1)\n    (RA / "bao_cao_cheo.json").write_text(json.dumps(bc, ensure_ascii=False, indent=2, default=str), encoding="utf-8")\n    print("KET QUA:", json.dumps({k: v for k, v in bc.items() if k != "cau_hinh"}, ensure_ascii=False, default=str)[:2000], flush=True)\n    if "loi" in bc:\n        raise RuntimeError("loi, xem bao_cao_cheo.json")\n', encoding='utf-8')


In [ ]:
import sys
sys.path.insert(0, '.')
import kaggle_cheo
kaggle_cheo.main(CAU_HINH)
